# 🛡️ Aegis — Benchmark Notebook

A copy of `aegis_demo.ipynb` with measurement code added. It runs the same DICOM and JPEG/PNG pipelines and records how fast they are, where the time goes, what they write to disk, and (when ground truth is available) how much PHI they remove.

| Measured | How |
|---|---|
| Throughput | Wall time, files/s, frames/s and MB/s per run, with model loading timed separately |
| Time per stage | Timers wrapped around file discovery, read, US masking, OCR, NER, header scrub and save. Aegis code is not modified. |
| Time per item | p50 / p90 / p99 / max per file or series, and the slowest items |
| Resources | Peak memory, CPU use and peak GPU memory, sampled during each run |
| File writes | Bytes and time for every output file, report CSV writes, review-queue copies, a raw disk baseline (local disk vs Drive), and a report-write strategy test |
| Accuracy (optional) | If `phi_overlays.csv` ground truth is found (from `synthetic_phi_generation.ipynb`): PHI boxes removed, non-PHI markers kept, PHI left in headers, PS3.15 verification |
| Projection | Estimated time and disk space for the full dataset |

**Before running:** put files in Google Drive the same way as for the demo:
```
My Drive/aegis_demo/dicom/   ← .dcm files (subfolders OK)
My Drive/aegis_demo/image/   ← .jpg / .png files (subfolders OK)
```
Then set the parameters in **Cell 3** and run the cells top to bottom. Results are saved in the benchmark folder (`bench_results/`) and copied to `My Drive/aegis_demo/benchmarks/`.

**Running outside Colab:** start Jupyter from inside the repository and set the `AEGIS_DEMO_DIR` environment variable to a folder that contains `dicom/` and `image/` (default: `staging_input/`). Nothing is installed in local mode.

> For a large dataset (for example 20K files), first run with `BENCH_MAX_FILES = 200` to check that everything works and to get a time estimate. Then run on everything.

## Cell 1 — Aegis setup (clone repo + install)

Set `AEGIS_BRANCH` to the branch you want to benchmark. NER name detection needs transformers 4.57.x; Cell 5 warns if a different major version is installed.

In [ ]:
import subprocess, sys, os

try:
    import google.colab  # noqa: F401  (only importable inside Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Which Aegis code to benchmark (Colab only; locally the current checkout is used) ──
AEGIS_REPO   = 'https://github.com/lakshmi-mahabaleshwara/aegis.git'
AEGIS_BRANCH = 'main'
# ───────────────────────────────────────────────────────────────────────────────


def _find_repo_root(start):
    d = os.path.abspath(start)
    while True:
        if os.path.isfile(os.path.join(d, 'pyproject.toml')) and os.path.isdir(os.path.join(d, 'src', 'monai_aegis')):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            raise RuntimeError('Start Jupyter from inside the aegis repository, or set AEGIS_ROOT.')
        d = parent


if IN_COLAB:
    AEGIS_ROOT = '/content/aegis'
    if not os.path.exists(AEGIS_ROOT):
        subprocess.run(['git', 'clone', '--branch', AEGIS_BRANCH, AEGIS_REPO, AEGIS_ROOT], check=True)
        print('Repo cloned.')
    else:
        subprocess.run(['git', '-C', AEGIS_ROOT, 'fetch', 'origin', AEGIS_BRANCH], check=True)
        subprocess.run(['git', '-C', AEGIS_ROOT, 'checkout', AEGIS_BRANCH], check=True)
        subprocess.run(['git', '-C', AEGIS_ROOT, 'pull', '--ff-only', 'origin', AEGIS_BRANCH], check=True)
        print('Repo updated.')
    os.chdir(AEGIS_ROOT)
    # Install third-party deps only; run monai_aegis straight from the cloned source (no pip -e).
    # pandas and psutil are for the measurements.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt',
                    'fsspec', 'pandas', 'psutil'], check=True)
else:
    AEGIS_ROOT = os.environ.get('AEGIS_ROOT') or _find_repo_root(os.getcwd())
    os.chdir(AEGIS_ROOT)
    print('Local run: using the current Python environment, nothing installed.')

# Put the src/ layout on the import path so `import monai_aegis` resolves from the checkout.
SRC = os.path.join(AEGIS_ROOT, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

try:
    import pandas  # noqa: F401
except ImportError:
    raise RuntimeError('pandas is required for the measurements: pip install pandas psutil')

import torch
gpu = torch.cuda.is_available()
print('Aegis root    :', AEGIS_ROOT)
print('GPU available :', gpu)
if gpu:
    print('  Device:', torch.cuda.get_device_name(0))
print('\nSetup complete.')

## Cell 2 — Mount Google Drive and locate the input

The demo also symlinked these folders into `staging_input/`. The runners read the Drive folder directly (it is the overlay's `input_dir`), so this copy skips the links.

In [ ]:
import os, glob, shutil, numpy as np, matplotlib.pyplot as plt, pydicom
from PIL import Image

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # ── Change this to match your Drive folder ───────────────────────
    DRIVE_DEMO_DIR = '/content/drive/MyDrive/aegis_demo'
    # ─────────────────────────────────────────────────────────────────
else:
    # A folder with dicom/ and image/ subfolders.
    DRIVE_DEMO_DIR = os.path.abspath(os.environ.get('AEGIS_DEMO_DIR', os.path.join(AEGIS_ROOT, 'staging_input')))

for sub in ('dicom', 'image'):
    p = os.path.join(DRIVE_DEMO_DIR, sub)
    if os.path.isdir(p):
        n = sum(1 for _, _, files in os.walk(p) for f in files if not f.startswith('.'))
        print('✅  {}  ({:,} files)'.format(p, n))
    else:
        print('❌  Not found: {}'.format(p))

## Cell 3 — Benchmark parameters

Change these before running the rest of the notebook.

In [ ]:
from datetime import datetime

# ── What to run ──────────────────────────────────────────────────────
BENCH_MAX_FILES   = None   # e.g. 200: benchmark a sample (whole folders) instead of every file
SAMPLE_SEED       = 0      # which folders the sample picks; same seed = same sample
STAGE_TO_LOCAL    = True   # copy the input to local disk first, so Drive read time is measured
                           # separately from processing (needs ~2x the input size free)
RUN_DICOM         = True   # DICOM series mode (what `aegis-pipeline --mode dicom` runs)
RUN_DICOM_SINGLE  = False  # also time per-file mode (OCR on every file) for comparison
RUN_IMAGE         = True   # JPEG/PNG pipeline
REPEATS           = 1      # >1 repeats each run to show run-to-run variation
NUM_WORKERS       = 0      # per-stage timers only see this process: keep 0 for stage numbers
QUIET_LOGS        = True   # hide per-file Aegis logs during runs (they slow a notebook down);
                           # errors are summarized in tables after each run instead
SAMPLE_INTERVAL_S = 0.5    # how often CPU / memory / GPU use is sampled

# ── File-write tests ─────────────────────────────────────────────────
DISK_TEST_MB       = 256     # large sequential write test, local disk
DRIVE_TEST_MB      = 32      # same test on Google Drive (Colab only; 0 = skip)
SMALL_FILE_COUNT   = 300     # many-small-files test: number of files...
SMALL_FILE_KB      = 512     # ...and the size of each
REPORT_BENCH_FILES = 20_000  # report-write strategy test: number of simulated files
REPORT_SERIES_SIZE = 100     # files per series for the "once per series" strategy

# ── Accuracy (needs phi_overlays.csv from synthetic_phi_generation.ipynb) ──
RUN_ACCURACY       = True
RUN_VERIFY         = True    # PS3.15 checklist over the output (reads every output file)
ACCURACY_MAX_FILES = 2_000   # most files checked pixel by pixel
ACCURACY_FRAMES    = 3       # frames checked per multi-frame file (first, middle, last)
INK_RESIDUAL_MAX   = 0.05    # a box counts as removed if at most 5% of its text pixels survive

# ── Projection ───────────────────────────────────────────────────────
TARGET_DATASET_FILES = 20_000

# ── Where results go ─────────────────────────────────────────────────
BENCH_ID   = datetime.now().strftime('%Y%m%d_%H%M%S')
BENCH_ROOT = '/content/aegis_bench' if IN_COLAB else os.path.join(AEGIS_ROOT, 'staging_output', 'benchmark')
BENCH_DIR  = os.path.join(BENCH_ROOT, BENCH_ID)
SAVE_RESULTS_TO_DRIVE = IN_COLAB

# Inputs are copied (STAGE_TO_LOCAL) or linked (sample only) into BENCH_DIR/input;
# otherwise the runners read the Drive folder directly.
STAGED         = STAGE_TO_LOCAL or BENCH_MAX_FILES is not None
RUN_INPUT_ROOT = os.path.join(BENCH_DIR, 'input') if STAGED else DRIVE_DEMO_DIR

os.makedirs(BENCH_DIR, exist_ok=True)
print('Benchmark folder:', BENCH_DIR)
print('Runner input    :', RUN_INPUT_ROOT, '(staged copy)' if STAGE_TO_LOCAL else '')

## Cell 4 — Load configuration (with overlay)

Each benchmark run writes its own overlay (input, output and review folders, worker count) into its run folder, so runs never mix their outputs. The overlay written here is the base one, used for display and for the warm-up.

In [ ]:
import torch, os, json, hashlib, yaml
from monai_aegis.config.config_loader import load_config

CONFIG_PATH   = os.path.join(AEGIS_ROOT, 'src', 'monai_aegis', 'config', 'config.yaml')
GPU_AVAILABLE = torch.cuda.is_available()


def write_overlay(path, input_dir, output_dir, not_processed_dir):
    """Write an Aegis config overlay for one run and return its path."""
    overlay = {
        'paths': {
            'input_dir': input_dir,
            'output_dir': output_dir,
            'not_processed_dir': not_processed_dir,
        },
        'runtime': {'dataloader_num_workers': NUM_WORKERS},
        'ocr': {'gpu_usage': GPU_AVAILABLE},
        'ner': {'device': 'cuda' if GPU_AVAILABLE else 'cpu'},
    }
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'w') as f:
        f.write('# Aegis benchmark overlay — auto-generated, do not edit manually\n')
        yaml.safe_dump(overlay, f, sort_keys=False)
    return path


OVERLAY_PATH = write_overlay(
    os.path.join(BENCH_DIR, 'overlays', 'base.yaml'),
    input_dir=DRIVE_DEMO_DIR,
    output_dir=os.path.join(BENCH_DIR, 'runs', '_base', 'output'),
    not_processed_dir=os.path.join(BENCH_DIR, 'runs', '_base', 'not_processed'),
)
os.environ['AEGIS_CONFIG_OVERRIDE'] = OVERLAY_PATH
config = load_config(CONFIG_PATH)

print('Overlay written to:', OVERLAY_PATH)
print('---')
with open(OVERLAY_PATH) as f:
    print(f.read())

print('=== Active config after overlay ===')
print('  input_dir :', config['paths']['input_dir'])
print('  output_dir:', config['paths']['output_dir'])
print('  gpu_usage :', config['ocr']['gpu_usage'])
print('  ner device:', config['ner']['device'])
print('  workers   :', config.get('runtime', {}).get('dataloader_num_workers'))

print('\n=== NER Settings ===')
ner = config.get('ner', {})
print('  enabled : {}'.format(ner.get('enabled')))
print('  model   : {}'.format(ner.get('model_name')))
labels = ner.get('phi_labels', [])
print('  PHI labels ({}) : {}'.format(len(labels), labels))

pii = config.get('pii_mapping', {})
print('\n=== PII Mapping — PS3.15 Basic Profile ({} tags) ==='.format(len(pii)))
for tag, action in list(pii.items())[:15]:
    print('  {} : {}'.format(tag, action))
if len(pii) > 15:
    print('  … and {} more'.format(len(pii) - 15))

# Fingerprint of the settings that change output (paths left out), to tell benchmark runs apart.
_cfg = {k: v for k, v in config.items() if k != 'paths'}
CONFIG_SHA256 = hashlib.sha256(json.dumps(_cfg, sort_keys=True, default=str).encode()).hexdigest()[:16]
print('\nConfig fingerprint:', CONFIG_SHA256)
print('\n✅  Config loaded with overlay applied')

## Cell 5 — Record the environment

Benchmark numbers only mean something next to the hardware and versions that produced them. This is saved with the results.

In [ ]:
import platform, subprocess, importlib.metadata as md
from datetime import timezone
import pandas as pd
from IPython.display import display

try:
    import psutil
except ImportError:
    psutil = None
    print('psutil is not installed: memory and CPU sampling are off (pip install psutil).')


def _ver(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


def _git(*args):
    try:
        return subprocess.run(['git', '-C', AEGIS_ROOT, *args], capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return None


def _cpu_model():
    try:
        with open('/proc/cpuinfo') as f:
            for line in f:
                if line.startswith('model name'):
                    return line.split(':', 1)[1].strip()
    except OSError:
        pass
    try:
        return subprocess.run(['sysctl', '-n', 'machdep.cpu.brand_string'],
                              capture_output=True, text=True, check=True).stdout.strip()
    except Exception:
        return platform.processor() or platform.machine()


ENV = {
    'bench_id': BENCH_ID,
    'started_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'in_colab': IN_COLAB,
    'python': platform.python_version(),
    'platform': platform.platform(),
    'cpu': _cpu_model(),
    'cpu_cores_logical': os.cpu_count(),
    'ram_gb': round(psutil.virtual_memory().total / 2**30, 1) if psutil else None,
    'gpu': torch.cuda.get_device_name(0) if GPU_AVAILABLE else None,
    'gpu_mem_gb': round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1) if GPU_AVAILABLE else None,
    'cuda': torch.version.cuda,
    'aegis_branch': _git('rev-parse', '--abbrev-ref', 'HEAD'),
    'aegis_commit': _git('rev-parse', '--short', 'HEAD'),
    'aegis_uncommitted_changes': bool(_git('status', '--porcelain', '--untracked-files=no')),
    'config_sha256': CONFIG_SHA256,
    'num_workers': NUM_WORKERS,
}
PACKAGES = ['torch', 'torchvision', 'transformers', 'tokenizers', 'huggingface_hub', 'easyocr', 'monai',
            'pydicom', 'numpy', 'opencv-python-headless', 'opencv-python', 'pillow', 'fsspec', 'pandas']
ENV['packages'] = {p: _ver(p) for p in PACKAGES if _ver(p)}

display(pd.Series({k: v for k, v in ENV.items() if k != 'packages'}, name='value').to_frame())
display(pd.Series(ENV['packages'], name='version').to_frame())

_tf = ENV['packages'].get('transformers')
if _tf and not _tf.startswith('4.57'):
    print(f'⚠️  transformers {_tf}: Aegis is validated on transformers 4.57.x. transformers 5 loads the '
          'NER tokenizer with the wrong casing, so patient names stop being detected and accuracy '
          'results will be too low. Benchmark a branch that pins transformers 4.57.')

## Cell 6 — Measurement harness

Defines the timers; nothing runs yet. During a benchmark run, these Aegis functions are temporarily wrapped with timers. The originals are put back when the run ends, even if it fails or is interrupted.

| Stage | Wraps | Inside |
|---|---|---|
| `discovery` | `discover_dicoms`, `discover_images` (reads every header) | runner |
| `pipeline_build` | `build_*pipeline` | runner |
| `load` | `LoadDicomRawd`, `LoadDicomSeriesd`, `LoadImaged`, `LoadImageSeriesd` (read + decode) | item |
| `us_regions` | `RedactByUSRegionsd` | item |
| `pixel_redact` | `RedactPixelPHId` (masks and drawing; OCR and NER timed separately) | item |
| `model_load_ocr` / `ocr` | `easyocr.Reader.__init__` / `readtext` | pixel_redact |
| `model_load_ner` / `ner` | `transformers.pipeline` / `PHIClassifier.classify_texts` | pixel_redact |
| `header_scrub` | `ScrubDicomMetadatad` | item |
| `save` / `file_write` | `SaveDicom*`, `SaveImage*` / `Dataset.save_as`, `Image.save` (encode + write) | item |
| `report_write` | report CSV writes | runner |
| `quarantine_copy` | copies into the review queue | runner |

An **item** is one pipeline call: a single file, or a whole series in series mode. Times are reported as **self time**: time in a stage minus time in the stages nested inside it. Self times therefore add up to the wall time without counting anything twice. Each timed call adds about a microsecond.

In [ ]:
import collections, csv, functools, gc, itertools, logging, random, re, threading, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom, PIL.Image, easyocr, transformers
from IPython.display import display
from monai_aegis import dicom_runner, image_runner, reporting
from monai_aegis.transforms import io as aegis_io, series_io, image_series_io, pixel, us_regions, ner_classifier
from monai_aegis.transforms.metadata import dict_transform

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

# ── Charts: one hue for single-series charts, recessive grid ─────────
SERIES_COLOR = '#257de0'
CHART_RC = {
    'axes.spines.top': False, 'axes.spines.right': False, 'axes.edgecolor': '#c8c8c8',
    'axes.grid': True, 'grid.color': '#ececec', 'axes.axisbelow': True,
    'axes.titlesize': 12, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left',
    'axes.labelcolor': '#444444', 'xtick.color': '#666666', 'ytick.color': '#666666',
}


class Recorder:
    """Everything measured during one benchmark run."""

    def __init__(self, name):
        self.name = name
        self.events = []    # one row per timed call
        self.items = []     # one row per pipeline item (a file, or a whole series)
        self.samples = []   # periodic CPU / memory / GPU samples
        self._tls = threading.local()

    @property
    def stack(self):
        if not hasattr(self._tls, 'stack'):
            self._tls.stack = []
        return self._tls.stack

    def enter(self, stage):
        self.stack.append([stage, 0.0])

    def exit(self, seconds):
        """Close the innermost call. Returns (parent stage, self seconds, inside an item?)."""
        stack = self.stack
        _, child_seconds = stack.pop()
        if stack:
            stack[-1][1] += seconds
        parent = stack[-1][0] if stack else None
        return parent, seconds - child_seconds, any(frame[0] == 'item' for frame in stack)


REC = None          # the active Recorder; None outside a benchmark run
RESULTS = {}        # run name -> measurements, filled by bench_run()
_ORIGINALS = {}     # (owner, attribute) -> original, while a run is in progress


def _timed(original, stage, extra=None):
    """Wrap `original` so each call during a run is recorded under `stage`."""
    @functools.wraps(original)
    def timed(*args, **kwargs):
        rec = REC
        if rec is None:
            return original(*args, **kwargs)
        rec.enter(stage)
        t0 = time.perf_counter()
        ok, result = False, None
        try:
            result = original(*args, **kwargs)
            ok = True
            return result
        finally:
            seconds = time.perf_counter() - t0
            parent, self_seconds, in_item = rec.exit(seconds)
            row = {'stage': stage, 'parent': parent, 'in_item': in_item,
                   'seconds': seconds, 'self_seconds': self_seconds, 'ok': ok}
            if extra is not None and ok:
                try:
                    row.update(extra(args, kwargs, result) or {})
                except Exception:
                    pass
            rec.events.append(row)
    return timed


def _root_error(data):
    """The original cause of a pipeline error. MONAI wraps it as 'applying transform ...'."""
    trace = data.get('error_trace') or ''
    causes = [l for l in trace.splitlines() if re.match(r'^[A-Za-z_][\w.]*(Error|Exception)\b[^:]*:', l)]
    msg = causes[0] if causes else str(data.get('error'))
    return re.sub(r' at 0x[0-9a-f]+', '', msg)[:200]


def _item_timer(runner):
    """Wrap a runner's _run_dataloader so each item (file or series) is timed."""
    def make(original):
        @functools.wraps(original)
        def run_dataloader(data_list, pipeline, num_workers):
            rec = REC
            it = iter(original(data_list, pipeline, num_workers))
            bar = tqdm(total=len(data_list), desc=rec.name, unit='item', leave=False) if (tqdm and rec) else None
            try:
                for idx in itertools.count():
                    if rec:
                        rec.enter('item')
                    t0 = time.perf_counter()
                    try:
                        data = next(it)
                    except StopIteration:
                        if rec:
                            rec.exit(time.perf_counter() - t0)
                        return
                    seconds = time.perf_counter() - t0
                    if rec:
                        _, unattributed, _ = rec.exit(seconds)
                        src = data_list[idx] if idx < len(data_list) else {}
                        uris = src.get('image') if isinstance(src, dict) else None
                        uris = list(uris) if isinstance(uris, (list, tuple)) else [uris]
                        err = _root_error(data) if isinstance(data, dict) and data.get('error') else None
                        rec.items.append({
                            'runner': runner, 'item': len(rec.items), 'files': len(uris),
                            'paths': [os.path.abspath(str(u)) for u in uris],
                            'seconds': seconds, 'unattributed_seconds': unattributed,
                            'error': bool(err), 'error_message': str(err)[:200] if err else None,
                        })
                    if bar:
                        bar.update(1)
                    yield data
            finally:
                if bar:
                    bar.close()
        return run_dataloader
    return make


# ── Extra fields recorded with some stages ───────────────────────────
def _size(path):
    try:
        return os.path.getsize(path)
    except (OSError, TypeError, ValueError):
        return None


def _write_extra(args, kwargs, result):
    # Dataset.save_as(self, filename, ...) and Image.save(self, fp, ...)
    target = args[1] if len(args) > 1 else kwargs.get('filename', kwargs.get('fp'))
    if isinstance(target, (str, os.PathLike)):
        path = os.fspath(target)
        return {'path': path, 'bytes': _size(path)}
    path = getattr(target, 'path', None) or getattr(target, 'name', None)
    try:
        nbytes = target.tell()   # bytes written so far into the open file
    except Exception:
        nbytes = None
    return {'path': str(path) if path else None, 'bytes': nbytes}


def _saved_extra(args, kwargs, result):
    paths = result if isinstance(result, (list, tuple)) else [result]
    sizes = [_size(p) for p in paths]
    return {'files': len(paths), 'bytes': sum(s for s in sizes if s)}


def _ocr_extra(args, kwargs, result):
    img = args[1] if len(args) > 1 else kwargs.get('image')
    shape = getattr(img, 'shape', None)
    return {'detections': len(result), 'ocr_input_shape': str(tuple(shape)) if shape is not None else None}


def _ner_extra(args, kwargs, result):
    texts = args[1] if len(args) > 1 else kwargs.get('texts', [])
    return {'texts': len(texts)}


def _rows_extra(args, kwargs, result):
    return {'rows': len(args[1]) if len(args) > 1 and args[1] is not None else 0}


def _quarantine_extra(args, kwargs, result):
    return {'path': str(args[0]), 'bytes': _size(args[0])}


WRAPPED = [
    # (owner, attribute, stage, extra fields)
    (dicom_runner, 'discover_dicoms', 'discovery', None),
    (image_runner, 'discover_images', 'discovery', None),
    (dicom_runner, 'build_pipeline', 'pipeline_build', None),
    (dicom_runner, 'build_series_pipeline', 'pipeline_build', None),
    (image_runner, 'build_image_pipeline', 'pipeline_build', None),
    (image_runner, 'build_image_series_pipeline', 'pipeline_build', None),
    (aegis_io.LoadDicomRawd, '__call__', 'load', None),
    (series_io.LoadDicomSeriesd, '__call__', 'load', None),
    (aegis_io.LoadImaged, '__call__', 'load', None),
    (image_series_io.LoadImageSeriesd, '__call__', 'load', None),
    (us_regions.RedactByUSRegionsd, '__call__', 'us_regions', None),
    (pixel.RedactPixelPHId, '__call__', 'pixel_redact', None),
    (easyocr.Reader, '__init__', 'model_load_ocr', None),
    (easyocr.Reader, 'readtext', 'ocr', _ocr_extra),
    (ner_classifier.PHIClassifier, 'classify_texts', 'ner', _ner_extra),
    (transformers, 'pipeline', 'model_load_ner', None),
    (dict_transform.ScrubDicomMetadatad, '__call__', 'header_scrub', None),
    (aegis_io.SaveDicom, '__call__', 'save', _saved_extra),
    (series_io.SaveDicomSeries, '__call__', 'save', _saved_extra),
    (aegis_io.SaveImage, '__call__', 'save', _saved_extra),
    (image_series_io.SaveImageSeries, '__call__', 'save', _saved_extra),
    (pydicom.dataset.Dataset, 'save_as', 'file_write', _write_extra),
    (PIL.Image.Image, 'save', 'file_write', _write_extra),
    (dicom_runner, '_quarantine_file', 'quarantine_copy', _quarantine_extra),
    (image_runner, '_quarantine_file', 'quarantine_copy', _quarantine_extra),
]
# Report writing differs between Aegis versions: rows streamed per file, or one write at the end.
if hasattr(reporting, '_CsvStream'):
    WRAPPED.append((reporting._CsvStream, 'write', 'report_write', _rows_extra))
for _fn in ('write_reports', 'write_phi_report'):
    if hasattr(reporting, _fn):
        WRAPPED.append((reporting, _fn, 'report_write', None))

MISSING = [f'{getattr(o, "__name__", o)}.{a}' for o, a, _, _ in WRAPPED if not hasattr(o, a)]
if MISSING:
    print('Not found in this Aegis version (those stages will show 0):', ', '.join(MISSING))


def _install(owner, name, make):
    if not hasattr(owner, name):
        return
    current = getattr(owner, name)
    original = getattr(current, '_bench_original', current)
    replacement = make(original)
    replacement._bench_original = original
    _ORIGINALS[(owner, name)] = original
    setattr(owner, name, replacement)


def _install_all():
    for owner, name, stage, extra in WRAPPED:
        _install(owner, name, lambda orig, s=stage, e=extra: _timed(orig, s, e))
    _install(dicom_runner, '_run_dataloader', _item_timer('dicom'))
    _install(image_runner, '_run_dataloader', _item_timer('image'))


def _uninstall_all():
    for (owner, name), original in _ORIGINALS.items():
        setattr(owner, name, original)
    _ORIGINALS.clear()


class ResourceSampler(threading.Thread):
    """Samples this process's memory and CPU use (and the GPU's) every `interval` seconds."""

    def __init__(self, rec, interval):
        super().__init__(daemon=True)
        self.rec, self.interval = rec, interval
        self.halt = threading.Event()
        self.proc = psutil.Process() if psutil else None
        self.gpu_util = GPU_AVAILABLE

    def run(self):
        t0 = time.perf_counter()
        if self.proc:
            self.proc.cpu_percent(None)
            psutil.cpu_percent(None)
        while not self.halt.wait(self.interval):
            s = {'t_s': time.perf_counter() - t0}
            if self.proc:
                s['rss_mb'] = self.proc.memory_info().rss / 2**20
                s['cpu_pct'] = self.proc.cpu_percent(None)        # 100 = one core fully busy
                s['system_cpu_pct'] = psutil.cpu_percent(None)    # whole machine, 0-100
            if GPU_AVAILABLE:
                s['gpu_mem_mb'] = torch.cuda.memory_allocated() / 2**20
                if self.gpu_util:
                    try:
                        s['gpu_util_pct'] = torch.cuda.utilization()
                    except Exception:
                        self.gpu_util = False
            self.rec.samples.append(s)

    def stop(self):
        self.halt.set()
        self.join(timeout=5)


def _io_counters():
    """OS-level read/write byte counters for this process (Linux only)."""
    if not psutil:
        return None
    try:
        return psutil.Process().io_counters()._asdict()
    except (AttributeError, NotImplementedError, psutil.Error):
        return None


def dir_stats(root):
    files, nbytes = 0, 0
    by_ext = collections.Counter()
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            size = _size(os.path.join(dirpath, fn)) or 0
            files += 1
            nbytes += size
            by_ext[os.path.splitext(fn)[1].lower() or '(none)'] += 1
    return {'files': files, 'bytes': nbytes, 'by_ext': dict(by_ext)}


def _run_output_dir(out_root, kind):
    folder = config['paths']['dicom_folder' if kind == 'dicom' else 'image_folder']
    dirs = sorted(glob.glob(os.path.join(out_root, folder, '*', '')))
    return dirs[-1].rstrip(os.sep) if dirs else None


def _q(series, q):
    return float(series.quantile(q)) if len(series) else None


def _summarize(rec, name, kind, status, wall, cpu0, cpu1, io0, io1, run_root, profile):
    ev = pd.DataFrame(rec.events)
    items = pd.DataFrame(rec.items)
    samples = pd.DataFrame(rec.samples)
    out_root = os.path.join(run_root, 'output')
    review_dir = os.path.join(run_root, 'not_processed')
    run_dir = _run_output_dir(out_root, kind)

    def total(stage, col='seconds', where=None):
        if ev.empty or stage not in set(ev['stage']):
            return 0.0
        sel = ev['stage'] == stage
        if where is not None:
            sel &= where
        return float(ev.loc[sel, col].fillna(0).sum())

    in_paths = [p for ps in items['paths'] for p in ps] if not items.empty else []
    in_files = len(in_paths)
    in_bytes = sum(_size(p) or 0 for p in in_paths)
    frames = pixels = None
    if profile is not None and not profile.empty and in_paths:
        fmap = dict(zip(profile['path'], profile['frames']))
        pmap = dict(zip(profile['path'], profile['frames'] * profile['rows'] * profile['cols']))
        frames = int(sum(fmap.get(p, 1) for p in in_paths))
        pixels = float(sum(pmap.get(p, 0) for p in in_paths))

    item_s = float(items['seconds'].sum()) if not items.empty else 0.0
    in_item = ev['in_item'] if not ev.empty else None
    model_load_in_items = (total('model_load_ocr', where=in_item) + total('model_load_ner', where=in_item)) if not ev.empty else 0.0
    processing_s = item_s - model_load_in_items
    out = dir_stats(out_root)
    review = dir_stats(review_dir)
    report_files = sorted(glob.glob(os.path.join(run_dir, 'aegis_*.csv'))) if run_dir else []
    cpu_s = (cpu1.user - cpu0.user) + (cpu1.system - cpu0.system)

    def per_s(n, s):
        return n / s if (n and s and s > 0) else None

    s = {
        'run': name, 'kind': kind, 'status': status,
        'items': len(items), 'input_files': in_files,
        'errors': int(items['error'].sum()) if not items.empty else 0,
        'input_mb': in_bytes / 2**20, 'frames': frames, 'megapixels': pixels / 1e6 if pixels else None,
        'wall_s': wall,
        'model_load_s': total('model_load_ocr') + total('model_load_ner'),
        'discovery_s': total('discovery'),
        'pipeline_build_s': total('pipeline_build'),
        'processing_s': processing_s,
        'files_per_s': per_s(in_files, processing_s),
        'files_per_s_wall': per_s(in_files, wall),
        'frames_per_s': per_s(frames, processing_s),
        'input_mb_per_s': per_s(in_bytes / 2**20, processing_s),
        's_per_file': processing_s / in_files if in_files else None,
        'item_p50_s': _q(items['seconds'], .5) if not items.empty else None,
        'item_p90_s': _q(items['seconds'], .9) if not items.empty else None,
        'item_p99_s': _q(items['seconds'], .99) if not items.empty else None,
        'item_max_s': float(items['seconds'].max()) if not items.empty else None,
        'ocr_calls': int((ev['stage'] == 'ocr').sum()) if not ev.empty else 0,
        'ocr_s': total('ocr'),
        'ner_s': total('ner', 'self_seconds'),
        'cpu_time_s': cpu_s,
        'cpu_cores_busy': cpu_s / wall if wall else None,
        'peak_rss_mb': float(samples['rss_mb'].max()) if 'rss_mb' in samples else None,
        'peak_gpu_mem_mb': torch.cuda.max_memory_allocated() / 2**20 if GPU_AVAILABLE else None,
        'output_files': out['files'], 'output_mb': out['bytes'] / 2**20,
        'review_files': review['files'], 'review_mb': review['bytes'] / 2**20,
        'file_write_files': int((ev['stage'] == 'file_write').sum()) if not ev.empty else 0,
        'file_write_mb': total('file_write', 'bytes') / 2**20 if not ev.empty and 'bytes' in ev else 0.0,
        'file_write_s': total('file_write'),
        'save_s': total('save'),
        'report_write_calls': int((ev['stage'] == 'report_write').sum()) if not ev.empty else 0,
        'report_write_s': total('report_write'),
        'report_mb': sum(_size(p) or 0 for p in report_files) / 2**20,
        'quarantine_copies': int((ev['stage'] == 'quarantine_copy').sum()) if not ev.empty else 0,
        'quarantine_s': total('quarantine_copy'),
        'run_dir': run_dir,
    }
    if io0 and io1:
        for key in ('read_bytes', 'write_bytes', 'read_chars', 'write_chars'):
            if key in io0:
                s[f'os_{key.replace("bytes", "disk").replace("chars", "calls")}_mb'] = (io1[key] - io0[key]) / 2**20
    return {'summary': s, 'events': ev, 'items': items, 'samples': samples,
            'run_dir': run_dir, 'run_root': run_root, 'review_dir': review_dir, 'output_stats': out}


def _save_run(result):
    mdir = os.path.join(result['run_root'], 'metrics')
    os.makedirs(mdir, exist_ok=True)
    with open(os.path.join(mdir, 'summary.json'), 'w') as f:
        json.dump(result['summary'], f, indent=2, default=str)
    for key in ('events', 'samples'):
        result[key].to_csv(os.path.join(mdir, f'{key}.csv'), index=False)
    items_table(result).to_csv(os.path.join(mdir, 'items.csv'), index=False)


def bench_run(name, kind, runner_fn, input_root, profile=None):
    """Run one Aegis runner with the timers installed; return its measurements.

    kind: 'dicom' or 'image'. profile: the input profile (adds frames per item).
    The result is kept in RESULTS[name] even if the run fails or is interrupted.
    """
    global REC
    if NUM_WORKERS:
        print(f'⚠️  NUM_WORKERS={NUM_WORKERS}: stages run in worker processes are not timed.')
    run_root = os.path.join(BENCH_DIR, 'runs', name)
    if os.path.isdir(run_root):
        shutil.rmtree(run_root)
    overlay = write_overlay(os.path.join(run_root, 'overlay.yaml'), input_root,
                            os.path.join(run_root, 'output'), os.path.join(run_root, 'not_processed'))
    os.environ['AEGIS_CONFIG_OVERRIDE'] = overlay

    rec = Recorder(name)
    sampler = ResourceSampler(rec, SAMPLE_INTERVAL_S)
    aegis_logger = logging.getLogger('monai_aegis')
    old_level = aegis_logger.level
    if QUIET_LOGS:
        aegis_logger.setLevel(logging.CRITICAL)
    if GPU_AVAILABLE:
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()
    gc.collect()
    io0, cpu0, status = _io_counters(), os.times(), 'ok'
    _install_all()
    REC = rec
    sampler.start()
    t0 = time.perf_counter()
    try:
        runner_fn(config_path=CONFIG_PATH)
    except BaseException as exc:
        status = f'{type(exc).__name__}: {exc}'[:300]
        raise
    finally:
        if GPU_AVAILABLE:
            torch.cuda.synchronize()
        wall = time.perf_counter() - t0
        REC = None
        _uninstall_all()
        sampler.stop()
        aegis_logger.setLevel(old_level)
        os.environ['AEGIS_CONFIG_OVERRIDE'] = OVERLAY_PATH
        result = _summarize(rec, name, kind, status, wall, cpu0, os.times(), io0, _io_counters(), run_root, profile)
        RESULTS[name] = result
        _save_run(result)
        s = result['summary']
        print(f"{name}: {status}. {s['input_files']:,} files in {wall:,.1f} s "
              f"({s['errors']} errors, {s['review_files']} sent to review)")
    return result


# ── Tables ───────────────────────────────────────────────────────────
STAGE_LABELS = {
    'discovery': 'Find input files (reads headers)',
    'pipeline_build': 'Build pipelines',
    'model_load_ocr': 'Load OCR models',
    'model_load_ner': 'Load NER model',
    'load': 'Read + decode input',
    'us_regions': 'US region masking',
    'ocr': 'OCR (EasyOCR)',
    'ner': 'Classify text (rules + NER)',
    'pixel_redact': 'Pixel redaction, other (masks, drawing)',
    'header_scrub': 'DICOM header scrub',
    'file_write': 'Encode + write output files',
    'save': 'Save, other (paths, folders, close)',
    'report_write': 'Write report CSVs',
    'quarantine_copy': 'Copy to review queue',
    'item': 'Pipeline, unattributed',
    'runner': 'Runner bookkeeping, unattributed',
}


def stage_table(result):
    """Time per stage. self_s adds up to the run's wall time."""
    ev, items, s = result['events'], result['items'], result['summary']
    rows = []
    if not ev.empty:
        for stage, grp in ev.groupby('stage'):
            rows.append({'stage': stage, 'calls': len(grp), 'total_s': grp['seconds'].sum(),
                         'self_s': grp['self_seconds'].sum(), 'mean_ms': grp['seconds'].mean() * 1e3,
                         'p95_ms': grp['seconds'].quantile(.95) * 1e3, 'max_ms': grp['seconds'].max() * 1e3})
    if not items.empty:
        rows.append({'stage': 'item', 'calls': len(items), 'total_s': items['seconds'].sum(),
                     'self_s': items['unattributed_seconds'].sum(), 'mean_ms': items['seconds'].mean() * 1e3,
                     'p95_ms': items['seconds'].quantile(.95) * 1e3, 'max_ms': items['seconds'].max() * 1e3})
    top_level = ev.loc[ev['parent'].isna(), 'seconds'].sum() if not ev.empty else 0.0
    item_total = items['seconds'].sum() if not items.empty else 0.0
    rows.append({'stage': 'runner', 'calls': None, 'total_s': s['wall_s'] - item_total - top_level,
                 'self_s': s['wall_s'] - item_total - top_level})
    df = pd.DataFrame(rows)
    df['share_pct'] = 100 * df['self_s'] / s['wall_s'] if s['wall_s'] else None
    df.insert(1, 'what', df['stage'].map(STAGE_LABELS).fillna(df['stage']))
    return df.sort_values('self_s', ascending=False).reset_index(drop=True)


def items_table(result, input_root=None):
    items = result['items']
    if items.empty:
        return items
    root = input_root or RUN_INPUT_ROOT
    df = items.drop(columns=['paths']).copy()
    df.insert(2, 'first_file', [os.path.relpath(ps[0], root) if ps and ps[0] else None for ps in items['paths']])
    return df


SUMMARY_FORMAT = [
    # key, label, format
    ('status', 'Status', '{}'),
    ('input_files', 'Input files', '{:,}'),
    ('items', 'Pipeline items (files or series)', '{:,}'),
    ('frames', 'Frames', '{:,}'),
    ('input_mb', 'Input size (MB)', '{:,.1f}'),
    ('wall_s', 'Wall time (s)', '{:,.1f}'),
    ('model_load_s', '  of which model loading (s)', '{:,.1f}'),
    ('discovery_s', '  of which finding files (s)', '{:,.1f}'),
    ('processing_s', '  of which processing items (s)', '{:,.1f}'),
    ('files_per_s', 'Files per second (processing)', '{:,.2f}'),
    ('files_per_s_wall', 'Files per second (wall)', '{:,.2f}'),
    ('frames_per_s', 'Frames per second (processing)', '{:,.2f}'),
    ('input_mb_per_s', 'Input MB per second (processing)', '{:,.2f}'),
    ('s_per_file', 'Seconds per file (processing)', '{:,.3f}'),
    ('item_p50_s', 'Item time p50 (s)', '{:,.3f}'),
    ('item_p90_s', 'Item time p90 (s)', '{:,.3f}'),
    ('item_p99_s', 'Item time p99 (s)', '{:,.3f}'),
    ('item_max_s', 'Item time max (s)', '{:,.3f}'),
    ('ocr_calls', 'OCR calls (frames read by OCR)', '{:,}'),
    ('cpu_cores_busy', 'CPU cores busy (average)', '{:,.2f}'),
    ('peak_rss_mb', 'Peak memory, RSS (MB)', '{:,.0f}'),
    ('peak_gpu_mem_mb', 'Peak GPU memory (MB)', '{:,.0f}'),
    ('errors', 'Errors', '{:,}'),
    ('output_files', 'Files written to output', '{:,}'),
    ('review_files', 'Files sent to review queue', '{:,}'),
]


def summary_table(result):
    s = result['summary']
    rows = [(label, fmt.format(s[k])) for k, label, fmt in SUMMARY_FORMAT if s.get(k) is not None]
    return pd.DataFrame(rows, columns=['metric', 'value']).set_index('metric')


# ── Charts ───────────────────────────────────────────────────────────
def bar_chart(labels, values, title, xlabel, fmt='{:,.1f}'):
    with plt.rc_context(CHART_RC):
        fig, ax = plt.subplots(figsize=(9, 0.4 * len(labels) + 1.3))
        y = np.arange(len(labels))
        ax.barh(y, values, color=SERIES_COLOR, height=0.6)
        ax.set_yticks(y)
        ax.set_yticklabels(labels)
        ax.invert_yaxis()
        ax.grid(axis='y', visible=False)
        ax.set_xlabel(xlabel)
        ax.set_title(title)
        for yi, v in zip(y, values):
            ax.text(v, yi, '  ' + fmt.format(v), va='center', fontsize=9, color='#444444')
        ax.set_xlim(0, (max(values) if len(values) else 1) * 1.2 or 1)
        plt.tight_layout()
        plt.show()


def histogram(values, title, xlabel):
    values = [v for v in values if v is not None and np.isfinite(v)]
    if len(values) < 2:
        return
    with plt.rc_context(CHART_RC):
        fig, ax = plt.subplots(figsize=(9, 3.2))
        ax.hist(values, bins=min(50, max(10, len(values) // 5)), color=SERIES_COLOR, edgecolor='white', linewidth=1)
        ax.grid(axis='x', visible=False)
        ax.set_xlabel(xlabel)
        ax.set_ylabel('count')
        ax.set_title(title)
        plt.tight_layout()
        plt.show()


def resource_chart(result):
    samples = result['samples']
    panels = [(c, t) for c, t in (('rss_mb', 'Memory, RSS (MB)'), ('cpu_pct', 'CPU (% of one core)'),
                                  ('gpu_mem_mb', 'GPU memory (MB)'), ('gpu_util_pct', 'GPU busy (%)'))
              if c in samples and samples[c].notna().any()]
    if samples.empty or not panels:
        return
    with plt.rc_context(CHART_RC):
        fig, axes = plt.subplots(len(panels), 1, figsize=(9, 2.2 * len(panels)), sharex=True, squeeze=False)
        for ax, (col, title) in zip(axes[:, 0], panels):
            ax.plot(samples['t_s'], samples[col], color=SERIES_COLOR, linewidth=2)
            ax.set_title(title)
            ax.set_ylim(bottom=0)
        axes[-1, 0].set_xlabel('seconds since run start')
        plt.tight_layout()
        plt.show()


def show_run(result):
    """Summary, time per stage, slowest items and resource use for one run."""
    s = result['summary']
    print(f"=== {s['run']} ===")
    display(summary_table(result))
    st = stage_table(result)
    display(st.style.format({'total_s': '{:,.2f}', 'self_s': '{:,.2f}', 'mean_ms': '{:,.1f}',
                             'p95_ms': '{:,.1f}', 'max_ms': '{:,.1f}', 'share_pct': '{:,.1f}',
                             'calls': '{:,.0f}'}, na_rep=''))
    shown = st[st['self_s'] > 0]
    bar_chart(list(shown['what']), list(shown['self_s']), f"Where the time went: {s['run']}",
              'seconds (self time)', '{:,.1f} s')
    items = items_table(result)
    if not items.empty:
        print('Slowest items:')
        display(items.sort_values('seconds', ascending=False).head(10)[['item', 'first_file', 'files', 'seconds', 'error']])
        histogram(list(items['seconds']), 'Time per item', 'seconds')
        errors = items[items['error']]
        if len(errors):
            print(f"⚠️  {len(errors):,} items ({int(errors['files'].sum()):,} files) failed and were sent to the review queue. By error:")
            display(errors.groupby('error_message').agg(items=('item', 'size'), files=('files', 'sum'),
                                                        example=('first_file', 'first')).sort_values('files', ascending=False))
    ev = result['events']
    if not ev.empty and 'ocr_input_shape' in ev:
        # What OCR was actually given. A shape that is not (height, width) or (height, width, 3)
        # of a real frame means the frames reached OCR with the wrong layout.
        shapes = ev.loc[ev['stage'] == 'ocr', 'ocr_input_shape'].value_counts().head(8)
        if len(shapes):
            print('Arrays passed to OCR (shape: calls):', ', '.join(f'{k}: {v:,}' for k, v in shapes.items()))
    resource_chart(result)


_OUTPUT_INDEX = {}


def output_for(input_path, result):
    """The output file Aegis wrote for input_path, matched by SOP Instance UID (DICOM), or None."""
    run_dir = result['run_dir']
    if not run_dir:
        return None
    if run_dir not in _OUTPUT_INDEX:
        src_to_deid = {}
        for name in ('aegis_tag_actions.csv', 'aegis_pixel_detections.csv'):
            p = os.path.join(run_dir, name)
            if os.path.exists(p):
                rep = pd.read_csv(p, usecols=['source_path', 'original_sop_uid', 'deid_sop_uid'], dtype=str).dropna()
                rep = rep.drop_duplicates(['original_sop_uid'])
                src_to_deid.update(zip(rep['source_path'].map(os.path.abspath), rep['deid_sop_uid']))
                src_to_deid.update(zip(rep['original_sop_uid'], rep['deid_sop_uid']))
        uid_to_out = {}
        for dirpath, _, filenames in os.walk(run_dir):
            for fn in filenames:
                p = os.path.join(dirpath, fn)
                if fn.startswith('aegis_') and fn.endswith('.csv'):
                    continue
                try:
                    uid_to_out[str(pydicom.dcmread(p, stop_before_pixels=True, specific_tags=['SOPInstanceUID']).SOPInstanceUID)] = p
                except Exception:
                    pass
        _OUTPUT_INDEX[run_dir] = (src_to_deid, uid_to_out)
    src_to_deid, uid_to_out = _OUTPUT_INDEX[run_dir]
    return uid_to_out.get(src_to_deid.get(input_path) or src_to_deid.get(os.path.abspath(input_path), ''))


print('Harness ready.')

## Cell 7 — Input helpers

Finding, sampling, staging and profiling input files. A sample takes **whole folders**, so series stay together (Aegis groups slices by series UID, and a series is usually one folder). Staging copies the sample to local disk and times the copy, which measures Drive read speed.

In [ ]:
import ast, re
from pydicom.uid import UID

try:
    from pydicom.pixels import get_decoder
except ImportError:      # pydicom < 3
    get_decoder = None


def _find_key(d, key):
    if isinstance(d, dict):
        if key in d:
            return d[key]
        for v in d.values():
            found = _find_key(v, key)
            if found is not None:
                return found
    return None


ACCEPTED_MODALITIES = set(_find_key(config, 'accepted_modalities') or [])


def list_dicoms(root):
    """DICOM files under root (by .dcm extension or DICM magic), plus any phi_overlays.csv."""
    files, gt_csvs = [], []
    for dirpath, _, fnames in os.walk(root):
        for f in sorted(fnames):
            if f.startswith('.'):
                continue
            fp = os.path.join(dirpath, f)
            if f == 'phi_overlays.csv':
                gt_csvs.append(fp)
                continue
            ext = os.path.splitext(f)[1].lower()
            if ext == '.dcm':
                files.append(os.path.relpath(fp, root))
            elif ext == '':
                try:
                    with open(fp, 'rb') as fh:
                        fh.seek(128)
                        if fh.read(4) == b'DICM':
                            files.append(os.path.relpath(fp, root))
                except OSError:
                    pass
    return sorted(files), sorted(gt_csvs)


def list_images(root):
    files = []
    for dirpath, _, fnames in os.walk(root):
        for f in sorted(fnames):
            if not f.startswith('.') and f.lower().endswith(('.jpg', '.jpeg', '.png')):
                files.append(os.path.relpath(os.path.join(dirpath, f), root))
    return sorted(files)


def pick_sample(rels, max_files, seed):
    """Whole folders, in a seeded random order, until max_files is reached."""
    if not max_files or len(rels) <= max_files:
        return list(rels)
    by_dir = collections.defaultdict(list)
    for r in rels:
        by_dir[os.path.dirname(r)].append(r)
    dirs = sorted(by_dir)
    random.Random(seed).shuffle(dirs)
    chosen = []
    for d in dirs:
        if len(chosen) >= max_files:
            break
        if not chosen and len(by_dir[d]) > max_files:
            print(f'⚠️  Folder {d!r} alone has {len(by_dir[d]):,} files; taking the first {max_files:,} (may split a series).')
            return sorted(by_dir[d])[:max_files]
        chosen.extend(by_dir[d])
    return sorted(chosen)


def prepare_inputs(folder, src_dir, rels):
    """Copy (STAGE_TO_LOCAL) or link (sample only) rels into RUN_INPUT_ROOT/folder; time the copy."""
    sizes = [_size(os.path.join(src_dir, r)) or 0 for r in rels]
    total = sum(sizes)
    free = shutil.disk_usage(BENCH_ROOT).free
    need = total * (2.2 if STAGE_TO_LOCAL else 1.1)    # input copy (if any) + output + margin
    print(f'Input {total / 2**30:,.2f} GB; free on benchmark disk {free / 2**30:,.1f} GB')
    if need > free:
        raise RuntimeError(f'Not enough disk space: need about {need / 2**30:,.1f} GB. '
                           'Set BENCH_MAX_FILES, or STAGE_TO_LOCAL = False.')
    if not STAGED:
        return {'mode': 'direct', 'files': len(rels), 'mb': total / 2**20}
    dst_dir = os.path.join(RUN_INPUT_ROOT, folder)
    if os.path.isdir(dst_dir):
        shutil.rmtree(dst_dir)
    mode = 'copy' if STAGE_TO_LOCAL else 'link'
    t0 = time.perf_counter()
    for r in (tqdm(rels, desc=f'{mode} {folder}', unit='file', leave=False) if tqdm else rels):
        src, dst = os.path.join(src_dir, r), os.path.join(dst_dir, r)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if STAGE_TO_LOCAL:
            shutil.copyfile(src, dst)
        else:
            os.symlink(src, dst)
    dt = time.perf_counter() - t0
    stats = {'mode': mode, 'files': len(rels), 'mb': total / 2**20, 'seconds': dt,
             'mb_per_s': total / 2**20 / dt if dt else None, 'files_per_s': len(rels) / dt if dt else None}
    if STAGE_TO_LOCAL:
        print(f"Copied {len(rels):,} files ({stats['mb']:,.1f} MB) from {src_dir} in {dt:,.1f} s: "
              f"{stats['mb_per_s']:,.1f} MB/s, {stats['files_per_s']:,.1f} files/s (source read speed)")
    return stats


PROFILE_TAGS = ['Modality', 'Rows', 'Columns', 'NumberOfFrames', 'SamplesPerPixel', 'BitsAllocated',
                'PhotometricInterpretation', 'SeriesInstanceUID', 'SOPInstanceUID', 'SequenceOfUltrasoundRegions']


def profile_dicoms(root, rels):
    """One row per file: size, modality, transfer syntax, frames, dimensions (header only)."""
    rows = []
    t0 = time.perf_counter()
    for r in (tqdm(rels, desc='profile dicom', unit='file', leave=False) if tqdm else rels):
        p = os.path.abspath(os.path.join(root, r))
        row = {'path': p, 'rel': r, 'bytes': _size(p)}
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True, specific_tags=PROFILE_TAGS)
            ts = str(getattr(getattr(ds, 'file_meta', None), 'TransferSyntaxUID', '') or '')
            row.update({
                'modality': str(getattr(ds, 'Modality', '') or ''),
                'transfer_syntax': ts, 'ts_name': UID(ts).name if ts else '',
                'frames': int(getattr(ds, 'NumberOfFrames', 1) or 1),
                'rows': int(getattr(ds, 'Rows', 0) or 0), 'cols': int(getattr(ds, 'Columns', 0) or 0),
                'samples_per_pixel': int(getattr(ds, 'SamplesPerPixel', 1) or 1),
                'photometric': str(getattr(ds, 'PhotometricInterpretation', '') or ''),
                'series_uid': str(getattr(ds, 'SeriesInstanceUID', '') or ''),
                'sop_uid': str(getattr(ds, 'SOPInstanceUID', '') or ''),
                'us_regions': 'SequenceOfUltrasoundRegions' in ds,
            })
        except Exception as exc:
            row['read_error'] = f'{type(exc).__name__}: {exc}'[:120]
        rows.append(row)
    df = pd.DataFrame(rows)
    df.attrs['scan_s'] = time.perf_counter() - t0
    return df


def _decodable(ts):
    if not ts or get_decoder is None:
        return None
    try:
        return bool(get_decoder(ts).is_available)
    except Exception:
        return False


def show_dicom_profile(df):
    if df.empty:
        print('No DICOM files.')
        return
    ok = df[df.get('read_error').isna()] if 'read_error' in df else df
    print(f"{len(df):,} files, {df['bytes'].sum() / 2**30:,.2f} GB, "
          f"{ok['series_uid'].nunique():,} series, {int(ok['frames'].sum()):,} frames "
          f"(header scan {df.attrs.get('scan_s', 0):,.1f} s)")
    print(f"File size MB: median {df['bytes'].median() / 2**20:,.2f}, max {df['bytes'].max() / 2**20:,.2f}")
    multi = ok[ok['frames'] > 1]
    if len(multi):
        kinds = ', '.join(f'{k}: {v:,}' for k, v in multi['photometric'].value_counts().items())
        print(f"Multi-frame files: {len(multi):,} ({kinds}; frames per file median "
              f"{int(multi['frames'].median())}, max {int(multi['frames'].max())})")
    display(ok.groupby('modality').agg(files=('path', 'size'), gb=('bytes', lambda b: b.sum() / 2**30),
                                       frames=('frames', 'sum'), us_regions=('us_regions', 'sum')))
    ts = ok.groupby(['transfer_syntax', 'ts_name']).size().rename('files').reset_index()
    ts['decodable_here'] = ts['transfer_syntax'].map(_decodable)
    display(ts)
    dims = (ok['cols'].astype(str) + ' x ' + ok['rows'].astype(str) + ' ' + ok['photometric']).value_counts().head(5)
    display(dims.rename('files').to_frame('files').rename_axis('width x height, colour'))
    # Things Aegis will not process: flag them before the run, not after.
    if ACCEPTED_MODALITIES:
        skipped = ok[~ok['modality'].isin(ACCEPTED_MODALITIES)]
        if len(skipped):
            print(f"⚠️  {len(skipped):,} files have a modality Aegis does not accept "
                  f"({', '.join(sorted(skipped['modality'].unique()))}); they will be skipped.")
    bad_ts = ts[ts['decodable_here'] == False]  # noqa: E712
    if len(bad_ts):
        print(f"⚠️  {int(bad_ts['files'].sum()):,} files use a compression this environment cannot "
              f"decode ({', '.join(bad_ts['ts_name'])}); they will fail.")
    if 'read_error' in df and df['read_error'].notna().any():
        print(f"⚠️  {int(df['read_error'].notna().sum()):,} files could not be read.")


def profile_images(root, rels):
    rows = []
    t0 = time.perf_counter()
    for r in (tqdm(rels, desc='profile image', unit='file', leave=False) if tqdm else rels):
        p = os.path.abspath(os.path.join(root, r))
        row = {'path': p, 'rel': r, 'bytes': _size(p), 'frames': 1}
        try:
            with PIL.Image.open(p) as im:      # reads the header only
                row.update({'format': im.format, 'mode': im.mode, 'cols': im.width, 'rows': im.height})
        except Exception as exc:
            row.update({'read_error': f'{type(exc).__name__}: {exc}'[:120], 'cols': 0, 'rows': 0})
        rows.append(row)
    df = pd.DataFrame(rows)
    df.attrs['scan_s'] = time.perf_counter() - t0
    return df


def show_image_profile(df):
    if df.empty:
        print('No image files.')
        return
    print(f"{len(df):,} files, {df['bytes'].sum() / 2**20:,.1f} MB (header scan {df.attrs.get('scan_s', 0):,.1f} s)")
    display(df.groupby(['format', 'mode']).size().rename('files').to_frame())
    dims = (df['cols'].astype(str) + ' x ' + df['rows'].astype(str)).value_counts().head(5)
    display(dims.rename('files').to_frame('files').rename_axis('width x height'))


def first_frame(ds):
    """The first frame of a DICOM, scaled to 0-255 for display."""
    arr = ds.pixel_array
    if int(getattr(ds, 'NumberOfFrames', 1) or 1) > 1:
        arr = arr[0]
    arr = arr.astype(float)
    return ((arr - arr.min()) / (arr.max() - arr.min() + 1e-5) * 255).astype('uint8')


print('Input helpers ready.')

## Cell 8 — Disk baseline (file write measurements, part 1)

Raw write speed of the disks involved, without Aegis, so pipeline write times can be compared against what the disk can do:

- **Large sequential write**: one big file, in 4 MB chunks. Buffered (the OS may still hold it in memory) and with `fsync` (forced to disk).
- **Many small files**: like writing one output file per slice.

On Colab the run output is on local disk (`/content`). Google Drive is tested too: Drive writes go to a local cache and upload later, so they look faster than Drive really is. Treat the Drive numbers as a best case.

In [ ]:
def write_test(dirpath, total_mb, fsync, chunk_mb=4):
    os.makedirs(dirpath, exist_ok=True)
    path = os.path.join(dirpath, f'_aegis_write_test_{os.getpid()}.bin')
    buf = os.urandom(chunk_mb * 2**20)
    n = max(1, total_mb // chunk_mb)
    t0 = time.perf_counter()
    with open(path, 'wb') as f:
        for _ in range(n):
            f.write(buf)
        f.flush()
        if fsync:
            os.fsync(f.fileno())
    dt = time.perf_counter() - t0
    os.remove(path)
    return {'mb': n * chunk_mb, 'seconds': dt, 'mb_per_s': n * chunk_mb / dt, 'files_per_s': None}


def small_files_test(dirpath, count, kb, fsync):
    d = os.path.join(dirpath, f'_aegis_small_files_{os.getpid()}')
    os.makedirs(d, exist_ok=True)
    buf = os.urandom(kb * 1024)
    t0 = time.perf_counter()
    for i in range(count):
        with open(os.path.join(d, f'{i:06d}.bin'), 'wb') as f:
            f.write(buf)
            if fsync:
                f.flush()
                os.fsync(f.fileno())
    dt = time.perf_counter() - t0
    shutil.rmtree(d)
    return {'mb': count * kb / 1024, 'seconds': dt, 'mb_per_s': count * kb / 1024 / dt, 'files_per_s': count / dt}


targets = [('local disk (output)', os.path.join(BENCH_DIR, '_disk_test'), DISK_TEST_MB, SMALL_FILE_COUNT)]
if IN_COLAB and DRIVE_TEST_MB:
    # Smaller on Drive: it uploads in the background and counts against your quota.
    targets.append(('Google Drive', os.path.join(DRIVE_DEMO_DIR, '_aegis_disk_test'), DRIVE_TEST_MB,
                    max(20, SMALL_FILE_COUNT // 10)))

storage_rows = []
for label, d, mb, count in targets:
    try:
        for fsync in (False, True):
            how = 'fsync' if fsync else 'buffered'
            storage_rows.append({'target': label, 'test': f'sequential {mb} MB, {how}', **write_test(d, mb, fsync)})
            storage_rows.append({'target': label, 'test': f'{count} x {SMALL_FILE_KB} KB files, {how}',
                                 **small_files_test(d, count, SMALL_FILE_KB, fsync)})
    finally:
        shutil.rmtree(d, ignore_errors=True)

STORAGE = pd.DataFrame(storage_rows)
display(STORAGE.style.format({'mb': '{:,.0f}', 'seconds': '{:,.2f}', 'mb_per_s': '{:,.1f}', 'files_per_s': '{:,.0f}'}, na_rep=''))
bar_chart([f"{r.target}: {r.test}" for r in STORAGE.itertuples()], list(STORAGE['mb_per_s']),
          'Raw disk write speed', 'MB/s', '{:,.0f} MB/s')

## Cell 9 — Warm-up (download and load the models once)

The first time the models are used they may be downloaded (EasyOCR weights and the Stanford NER model). Doing it here keeps download time out of the timed runs, which then measure loading from local disk.

In [ ]:
from monai_aegis.transforms.ner_classifier import PHIClassifier
try:
    from monai_aegis.config.config_loader import as_bool
except ImportError:
    def as_bool(v, default=False):
        return default if v is None else str(v).strip().lower() in ('1', 'true', 'yes', 'on')

ocr_cfg = config['ocr']
t0 = time.perf_counter()
_reader = easyocr.Reader(ocr_cfg.get('languages', ['en']), gpu=GPU_AVAILABLE,
                         model_storage_directory=ocr_cfg.get('model_storage_directory') or None,
                         download_enabled=as_bool(ocr_cfg.get('download_enabled'), default=True))
t_ocr = time.perf_counter() - t0
t0 = time.perf_counter()
_ = PHIClassifier(config).pipeline
t_ner = time.perf_counter() - t0
WARMUP = {'ocr_first_load_s': t_ocr, 'ner_first_load_s': t_ner}
del _reader, _
gc.collect()
if GPU_AVAILABLE:
    torch.cuda.empty_cache()
print(f'OCR models: {t_ocr:,.1f} s   NER model: {t_ner:,.1f} s   (first load, including any download)')

---
## -------------- PART A: DICOM -------------------

## Cell 10 — Find, sample and profile DICOM input

Lists the DICOM files, takes the sample (if `BENCH_MAX_FILES` is set), stages it, then reads every header to profile the dataset. The profile flags files Aegis will skip or fail on **before** the run starts: modalities outside `accepted_modalities`, and compression this environment cannot decode.

In [ ]:
DICOM_FOLDER    = config['paths']['dicom_folder']
DICOM_INPUT_DIR = os.path.join(DRIVE_DEMO_DIR, DICOM_FOLDER)

t0 = time.perf_counter()
dcm_all, GT_CSVS = list_dicoms(DICOM_INPUT_DIR)
print('Found {:,} DICOM file(s) in {} ({:,.1f} s to list)'.format(len(dcm_all), DICOM_INPUT_DIR, time.perf_counter() - t0))
if GT_CSVS:
    print(f'Found {len(GT_CSVS):,} phi_overlays.csv ground-truth file(s): the accuracy check (Cell 18) can run.')

dcm_files = pick_sample(dcm_all, BENCH_MAX_FILES, SAMPLE_SEED)
if len(dcm_files) < len(dcm_all):
    print('Sample: {:,} files from {:,} folders'.format(len(dcm_files), len({os.path.dirname(f) for f in dcm_files})))

DICOM_STAGE   = prepare_inputs(DICOM_FOLDER, DICOM_INPUT_DIR, dcm_files) if dcm_files else {}
RUN_DICOM_DIR = os.path.join(RUN_INPUT_ROOT, DICOM_FOLDER)
DICOM_PROFILE = profile_dicoms(RUN_DICOM_DIR, dcm_files)
show_dicom_profile(DICOM_PROFILE)

print()
for f in dcm_files[:20]:
    print('  {:9.1f} KB  {}'.format((_size(os.path.join(RUN_DICOM_DIR, f)) or 0) / 1024, f))
if len(dcm_files) > 20:
    print('  … and {:,} more'.format(len(dcm_files) - 20))
if not dcm_files:
    print('No DICOM files found. Add .dcm files to Drive and re-run.')

## Cell 11 — Preview BEFORE (PHI visible)

In [ ]:
if not dcm_files:
    raise RuntimeError('No DICOM files found. Run Cell 10 first.')

DCM_SAMPLE      = dcm_files[0]
dcm_sample_path = os.path.join(RUN_DICOM_DIR, DCM_SAMPLE)
print('Using:', DCM_SAMPLE)

ds_before = pydicom.dcmread(dcm_sample_path)
arr = first_frame(ds_before)

plt.figure(figsize=(10, 8))
plt.imshow(arr, cmap='gray' if arr.ndim == 2 else None)
plt.title('BEFORE — {} (PHI visible)'.format(DCM_SAMPLE), fontsize=13, color='red')
plt.axis('off'); plt.tight_layout(); plt.show()

print('\nRaw DICOM tags before de-identification:')
for field in ['PatientName', 'PatientID', 'PatientBirthDate', 'StudyDate', 'InstitutionName', 'Modality']:
    print('  {:<20}: {}'.format(field, getattr(ds_before, field, 'N/A')))

## Cell 12 — Run the DICOM benchmark

Runs `dicom_runner.run_series` (and `run_single` if `RUN_DICOM_SINGLE`), `REPEATS` times each, with the timers installed. Each run writes to its own folder under `runs/`. If you interrupt a run, everything measured up to that point is kept in `RESULTS`.

In [ ]:
print('Running dicom_runner...\n')
DICOM_RUNS = []
if RUN_DICOM and dcm_files:
    for r in range(1, REPEATS + 1):
        DICOM_RUNS.append(bench_run(f'dicom_series_{r}', 'dicom', dicom_runner.run_series, RUN_INPUT_ROOT, DICOM_PROFILE))
if RUN_DICOM_SINGLE and dcm_files:
    for r in range(1, REPEATS + 1):
        DICOM_RUNS.append(bench_run(f'dicom_single_{r}', 'dicom', dicom_runner.run_single, RUN_INPUT_ROOT, DICOM_PROFILE))

DICOM_RESULT = DICOM_RUNS[0] if DICOM_RUNS else None
RUN_DIR = DICOM_RESULT['run_dir'] if DICOM_RESULT else None
print('\n✅  DICOM runner complete' if DICOM_RESULT else 'DICOM benchmark skipped.')

## Cell 13 — DICOM results: throughput, time per stage, resources

How to read it:
- **Files per second (processing)** leaves out model loading, which happens once per run. Use it to estimate large runs.
- **Where the time went** uses self time, so the bars add up to the wall time.
- **OCR calls** is the number of frames OCR actually read. In series mode only keyframes are read, so it is lower than the frame count.

In [ ]:
for res in DICOM_RUNS:
    show_run(res)

if len(DICOM_RUNS) > 1:
    cols = ['run', 'input_files', 'wall_s', 'model_load_s', 'processing_s', 'files_per_s',
            'frames_per_s', 'ocr_calls', 'peak_rss_mb', 'peak_gpu_mem_mb']
    print('=== All DICOM runs ===')
    display(pd.DataFrame([r['summary'] for r in DICOM_RUNS])[cols].set_index('run'))

## Cell 14 — DICOM file write measurements (part 2)

What the pipeline wrote and how long it took:

- **Per output file**: bytes and time for `Dataset.save_as` (encode + write) on every output file.
- **Save, total**: the whole save step, which also creates folders and closes files.
- **Report CSVs**: time spent writing `aegis_pixel_detections.csv` and `aegis_tag_actions.csv`.
- **Review queue**: copies of files sent for manual review.
- **OS counters** (Linux only): bytes passed to `write()` calls, and bytes that reached the disk.

Compare **aggregate MB/s** with the disk baseline in Cell 8. If they are close, the disk limits writing. If much lower, encoding or many small files do.

In [ ]:
def write_tables(result):
    s, ev = result['summary'], result['events']
    fw = ev[ev['stage'] == 'file_write'].copy() if not ev.empty else pd.DataFrame()
    per_file = pd.DataFrame()
    if not fw.empty:
        root = result['run_dir'] or result['run_root']
        per_file = pd.DataFrame({
            'file': [os.path.relpath(p, root) if isinstance(p, str) and p.startswith(root) else p for p in fw['path']],
            'mb': fw['bytes'].astype(float) / 2**20,
            'ms': fw['seconds'] * 1e3,
        })
        per_file['mb_per_s'] = per_file['mb'] / (per_file['ms'] / 1e3)
    rows = [
        ('Output files written', f"{s['file_write_files']:,}"),
        ('Output written (MB)', f"{s['file_write_mb']:,.1f}"),
        ('Output vs input size', f"{s['output_mb'] / s['input_mb']:,.2f}x" if s['input_mb'] else ''),
        ('Encode + write time (s)', f"{s['file_write_s']:,.2f}"),
        ('Save step total, incl. folders and close (s)', f"{s['save_s']:,.2f}"),
        ('Write share of wall time', f"{100 * s['save_s'] / s['wall_s']:,.1f}%" if s['wall_s'] else ''),
    ]
    if not per_file.empty:
        rows += [
            ('Aggregate write speed (MB/s)', f"{per_file['mb'].sum() / (per_file['ms'].sum() / 1e3):,.1f}"),
            ('Per file: mean / p50 / p95 / max (ms)', ' / '.join(f'{v:,.1f}' for v in (
                per_file['ms'].mean(), per_file['ms'].median(), per_file['ms'].quantile(.95), per_file['ms'].max()))),
            ('Per file size: median / max (MB)', f"{per_file['mb'].median():,.2f} / {per_file['mb'].max():,.2f}"),
        ]
    rows += [
        ('Report CSV writes (calls)', f"{s['report_write_calls']:,}"),
        ('Report CSV write time (s)', f"{s['report_write_s']:,.3f}"),
        ('Report CSV size (MB)', f"{s['report_mb']:,.2f}"),
        ('Review-queue copies (files)', f"{s['quarantine_copies']:,}"),
        ('Review-queue copy time (s)', f"{s['quarantine_s']:,.2f}"),
        ('Review queue size (MB)', f"{s['review_mb']:,.1f}"),
    ]
    for key, label in (('os_write_calls_mb', 'OS: bytes passed to write() (MB)'),
                       ('os_write_disk_mb', 'OS: bytes that reached the disk (MB)'),
                       ('os_read_calls_mb', 'OS: bytes read (MB)')):
        if s.get(key) is not None:
            rows.append((label, f'{s[key]:,.1f}'))
    return pd.DataFrame(rows, columns=['metric', 'value']).set_index('metric'), per_file


def show_writes(result):
    print(f"=== File writes: {result['summary']['run']} ===")
    table, per_file = write_tables(result)
    display(table)
    if not per_file.empty:
        print('Slowest output writes:')
        display(per_file.sort_values('ms', ascending=False).head(10).style.format(
            {'mb': '{:,.2f}', 'ms': '{:,.1f}', 'mb_per_s': '{:,.1f}'}))
        histogram(list(per_file['ms']), 'Time to encode + write one output file', 'milliseconds')
    by_ext = result['output_stats']['by_ext']
    if by_ext:
        print('Files in the output folder by type:', ', '.join(f'{k}: {v:,}' for k, v in sorted(by_ext.items())))


for res in DICOM_RUNS:
    show_writes(res)

## Cell 15 — Redaction stats + DICOM metadata before / after

In [ ]:
if not DICOM_RESULT:
    raise RuntimeError('No DICOM run. Run Cell 12 first.')

output_dcm_files = sorted(glob.glob(os.path.join(RUN_DIR, '**', '*.dcm'), recursive=True)) if RUN_DIR else []
quarantined      = sorted(glob.glob(os.path.join(DICOM_RESULT['review_dir'], '**', '*'), recursive=True))
quarantined      = [q for q in quarantined if os.path.isfile(q)]

print('=== Pipeline Run Summary ===')
print('  Output files     :', len(output_dcm_files))
print('  Quarantined files:', len(quarantined))
print('  Run folder       :', RUN_DIR)

if quarantined:
    print('\n⚠️  Quarantined (low-confidence / errors):')
    for f in quarantined[:20]:
        print('    {}'.format(os.path.relpath(f, DICOM_RESULT['review_dir'])))
    if len(quarantined) > 20:
        print('    … and {:,} more'.format(len(quarantined) - 20))
else:
    print('\n✅  No files quarantined')

# Find the sample's output by SOP Instance UID (output folders can be tokenized and file
# names repeat across series). If the sample went to the review queue, use the first file
# that was processed instead, so BEFORE and AFTER are always the same image.
SCRUBBED_DCM_PATH = output_for(dcm_sample_path, DICOM_RESULT)
if SCRUBBED_DCM_PATH is None:
    for f in dcm_files:
        out = output_for(os.path.join(RUN_DICOM_DIR, f), DICOM_RESULT)
        if out:
            print(f'\n⚠️  {DCM_SAMPLE} is not in the output (review queue or error). Using {f} instead.')
            DCM_SAMPLE, dcm_sample_path, SCRUBBED_DCM_PATH = f, os.path.join(RUN_DICOM_DIR, f), out
            ds_before = pydicom.dcmread(dcm_sample_path)
            break

pii_tags = [
    ('(0010,0010)', 'PatientName'),
    ('(0010,0020)', 'PatientID'),
    ('(0010,0030)', 'PatientBirthDate'),
    ('(0008,0020)', 'StudyDate'),
    ('(0008,0050)', 'AccessionNumber'),
    ('(0008,0090)', 'ReferringPhysicianName'),
    ('(0008,0080)', 'InstitutionName'),
    ('(0008,0021)', 'SeriesDate'),
    ('(0008,0023)', 'ContentDate'),
]


def _tag_val(ds, tag_str):
    parts = tag_str.strip('()').split(',')
    tag = pydicom.tag.Tag(int(parts[0].strip(), 16), int(parts[1].strip(), 16))
    return str(ds[tag].value) if tag in ds else '<REMOVED>'


if not SCRUBBED_DCM_PATH:
    print('\nNo output DICOM found — check runner output above.')
else:
    ds_after = pydicom.dcmread(SCRUBBED_DCM_PATH)
    print('\n=== DICOM Metadata: Before vs After ===')
    print('{:<18} {:<26} {:<32} {}'.format('Tag', 'Field', 'BEFORE', 'AFTER'))
    print('-' * 100)
    changed = unchanged = 0
    for tag, name in pii_tags:
        before = _tag_val(ds_before, tag)
        after = _tag_val(ds_after, tag)
        if before != after:
            marker, changed = '🔴', changed + 1
        else:
            marker, unchanged = '⚪', unchanged + 1
        print('{} {:<16} {:<26} {:<32} {}'.format(marker, tag, name, before[:30], after))
    print('\n  🔴 Changed  : {}'.format(changed))
    print('  ⚪ Unchanged: {}'.format(unchanged))

## Cell 16 — Ground-truth reports (detections + tag actions)

When `reporting.save_ground_truth: true` in `config.yaml` (the default), every run writes two CSVs into the timestamped output folder: Aegis's own record of what it detected and redacted.

- **`aegis_pixel_detections.csv`**: one row per OCR region (burnt-in pixel PHI): original/de-identified `SOPInstanceUID`, bounding box, frame index, tokenized text, confidence, and decision (`redacted` / `safelisted` / `low_confidence`).
- **`aegis_tag_actions.csv`**: one row per scrubbed DICOM header tag: tag, keyword, action (`REMOVE` / `DUMMY` / `ZERO`), and the source/output UIDs.

Both files keep the **original `SOPInstanceUID`**, so they join directly to synthetic or manual ground truth (Cell 18 does this).

In [ ]:
print('Run directory:', RUN_DIR)
if RUN_DIR:
    pixel_csv = os.path.join(RUN_DIR, 'aegis_pixel_detections.csv')
    tag_csv = os.path.join(RUN_DIR, 'aegis_tag_actions.csv')

    print('\n=== aegis_pixel_detections.csv (burnt-in pixel PHI) ===')
    if os.path.exists(pixel_csv):
        pix = pd.read_csv(pixel_csv)
        print('rows:', len(pix))
        if len(pix):
            print('decisions:', pix['decision'].value_counts().to_dict())
            print('rows per input file: {:,.1f}'.format(len(pix) / max(1, DICOM_RESULT['summary']['input_files'])))
        display(pix.head(10))
    else:
        print('  (not found — is reporting.save_ground_truth enabled?)')

    print('\n=== aegis_tag_actions.csv (header metadata scrubbing) ===')
    if os.path.exists(tag_csv):
        tags = pd.read_csv(tag_csv)
        print('rows:', len(tags))
        if len(tags):
            print('actions:', tags['action'].value_counts().to_dict())
        display(tags.head(15))
    else:
        print('  (not found — is reporting.save_ground_truth enabled?)')

## Cell 17 — US region masking + BEFORE / AFTER side by side

In [ ]:
# Load the sample DICOM fresh to inspect US regions
ds = pydicom.dcmread(dcm_sample_path)
modality = getattr(ds, 'Modality', 'N/A')
print('Modality:', modality)

if (0x0018, 0x6011) in ds:
    regions = ds[0x0018, 0x6011].value
    print('SequenceOfUltrasoundRegions — {} region(s) found:'.format(len(regions)))
    for i, r in enumerate(regions):
        x0 = getattr(r, 'RegionLocationMinX0', 'N/A')
        y0 = getattr(r, 'RegionLocationMinY0', 'N/A')
        x1 = getattr(r, 'RegionLocationMaxX1', 'N/A')
        y1 = getattr(r, 'RegionLocationMaxY1', 'N/A')
        fmt = getattr(r, 'RegionSpatialFormat', 'N/A')
        print('  Region {}: ({},{}) → ({},{})  spatial_format={}'.format(i, x0, y0, x1, y1, fmt))

    from monai_aegis.transforms.us_regions import build_us_phi_mask
    orig = first_frame(ds)
    H, W = orig.shape[:2]
    mask = build_us_phi_mask(ds, H, W)
    if mask is not None:
        phi_pct = int(100 * mask.sum() / mask.size)
        print('\nPHI zone (OCR scanned)    : {}% of frame'.format(phi_pct))
        print('Diagnostic fan (protected): {}% of frame'.format(100 - phi_pct))
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        axes[0].imshow(orig, cmap='gray' if orig.ndim == 2 else None)
        axes[0].set_title('Original frame'); axes[0].axis('off')
        axes[1].imshow(mask, cmap='hot')
        axes[1].set_title('US PHI mask  (bright = OCR zone)'); axes[1].axis('off')
        plt.suptitle('SequenceOfUltrasoundRegions — Fan geometry', fontsize=13, fontweight='bold')
        plt.tight_layout(); plt.show()
else:
    print('No (0018,6011) tag — not a tagged US file.')
    print('RedactByUSRegionsd was a no-op; full-frame OCR ran on this file.')

print('\nBefore file :', dcm_sample_path)
print('After file  :', SCRUBBED_DCM_PATH)
if SCRUBBED_DCM_PATH:
    ds_after = pydicom.dcmread(SCRUBBED_DCM_PATH)
    orig_display, after_display = first_frame(ds_before), first_frame(ds_after)
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    axes[0].imshow(orig_display, cmap='gray' if orig_display.ndim == 2 else None)
    axes[0].set_title('BEFORE — PHI visible', fontsize=14, color='red', fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(after_display, cmap='gray' if after_display.ndim == 2 else None)
    axes[1].set_title('AFTER — PHI redacted by Aegis', fontsize=14, color='green', fontweight='bold')
    axes[1].axis('off')
    plt.suptitle('Aegis De-identification  |  {}'.format(DCM_SAMPLE), fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()

## Cell 18 — Accuracy against ground truth (optional)

Runs only if `phi_overlays.csv` files were found (Cell 10). Those come from `synthetic_phi_generation.ipynb` and list every box burnt into each file, with `is_phi` marking PHI (name, MRN, date/time) versus non-PHI markers (manufacturer, scanner settings, depth ruler).

Each box is checked on the pixels themselves, independently of Aegis's own report. The check finds the box's text pixels in the input, then counts how many are unchanged in the output (the **residual**). It looks at the first, middle and last frame and keeps the worst result.

- **PHI box removed**: residual at most `INK_RESIDUAL_MAX` (5%) on every checked frame.
- **Non-PHI marker kept**: residual at least 95%. Anything removed here is over-redaction.
- **File clean**: every PHI box in the file removed.

It also checks whether the ground-truth name, MRN and study date still appear anywhere in the output header, and runs the PS3.15 checklist (`aegis-verify`) over the run folder. Result tables contain file paths and categories, never the PHI values.

In [ ]:
ACCURACY = {}


def load_ground_truth(paths, src_root):
    frames = [pd.read_csv(p).assign(gt_file=os.path.relpath(p, src_root)) for p in paths]
    gt = pd.concat(frames, ignore_index=True)
    if {'bbox_x', 'bbox_y', 'bbox_w', 'bbox_h'} <= set(gt.columns):
        gt['box'] = list(zip(gt['bbox_x'], gt['bbox_y'], gt['bbox_w'], gt['bbox_h']))
    else:
        gt['box'] = gt['bbox'].map(lambda v: tuple(int(x) for x in ast.literal_eval(str(v))))
    if 'is_phi' in gt:
        gt['is_phi'] = gt['is_phi'].astype(str).str.lower().isin(['true', '1'])
    else:
        gt['is_phi'] = gt['phi_category'].isin(['patient_name', 'mrn', 'study_datetime'])
    gt['sop_instance_uid'] = gt['sop_instance_uid'].astype(str)
    return gt


def _read_frames(path, indices):
    try:
        from pydicom.pixels import pixel_array as _px
        return {i: _px(path, index=i) for i in indices}
    except Exception:
        ds = pydicom.dcmread(path)
        arr = ds.pixel_array
        multi = int(getattr(ds, 'NumberOfFrames', 1) or 1) > 1
        return {i: (arr[i] if multi else arr) for i in indices}


def _gray(a):
    return a.max(axis=-1) if a.ndim == 3 else a


def residual_ink(before, after, box, pad=3):
    """Share of the box's text pixels that are unchanged in the output (0 = all removed)."""
    x, y, w, h = (int(v) for v in box)
    b, a = _gray(before).astype(np.int32), _gray(after).astype(np.int32)
    inner_b, inner_a = b[y:y + h, x:x + w], a[y:y + h, x:x + w]
    if inner_b.size == 0:
        return np.nan
    outer = b[max(y - pad, 0):y + h + pad, max(x - pad, 0):x + w + pad].copy()
    ring = np.ones(outer.shape, bool)
    ring[min(pad, y):min(pad, y) + h, min(pad, x):min(pad, x) + w] = False
    background = np.median(outer[ring]) if ring.any() else np.median(inner_b)
    ink = np.abs(inner_b - background) > 40
    if not ink.any():
        return np.nan
    return float((np.abs(inner_a - inner_b)[ink] <= 3).mean())


def _header_text(ds):
    parts = []
    for el in ds.iterall():
        if el.tag == 0x7FE00010 or el.VR in ('SQ', 'OB', 'OW', 'OF', 'OD', 'OL', 'UN'):
            continue
        if el.value is not None and not isinstance(el.value, (bytes, bytearray)):
            parts.append(str(el.value))
    return ' '.join(parts).lower()


def _header_tokens(category, value):
    value = str(value)
    if category == 'patient_name':
        return [t for t in re.findall(r'[A-Za-z]{3,}', value)]
    if category == 'mrn':
        return re.findall(r'\d{4,}', value)
    if category == 'study_datetime':
        return [value[:8]] if re.fullmatch(r'\d{8,}', value) else []
    return []


def evaluate_accuracy(result, profile, gt_csvs):
    gt = load_ground_truth(gt_csvs, DICOM_INPUT_DIR)
    in_path = dict(zip(profile['sop_uid'], profile['path']))
    gt = gt[gt['sop_instance_uid'].isin(in_path)]          # files in this benchmark's sample
    if gt.empty:
        print('No ground-truth boxes belong to the files in this run.')
        return None
    run_dir = result['run_dir']
    sops = sorted(gt['sop_instance_uid'].unique())[:ACCURACY_MAX_FILES]
    box_rows, file_rows = [], []
    t0 = time.perf_counter()
    for sop in (tqdm(sops, desc='accuracy', unit='file', leave=False) if tqdm else sops):
        boxes = gt[gt['sop_instance_uid'] == sop]
        src = in_path[sop]
        rel = os.path.relpath(src, RUN_DICOM_DIR)
        dst = output_for(sop, result)      # original SOP UID -> de-identified UID -> output file
        if dst is None:
            file_rows.append({'file': rel, 'status': 'not in output (review queue or error)'})
            continue
        n = int(profile.loc[profile['sop_uid'] == sop, 'frames'].iloc[0])
        idx = sorted({0, n // 2, n - 1})[:ACCURACY_FRAMES]
        before, after = _read_frames(src, idx), _read_frames(dst, idx)
        if any(before[i].shape != after[i].shape for i in idx):
            file_rows.append({'file': rel, 'status': 'output shape differs'})
            continue
        for b in boxes.itertuples():
            res = [residual_ink(before[i], after[i], b.box) for i in idx]
            res = [r for r in res if not np.isnan(r)]
            worst = (max(res) if b.is_phi else min(res)) if res else np.nan
            if np.isnan(worst):
                status = 'no text pixels found'
            elif worst <= INK_RESIDUAL_MAX:
                status = 'removed'
            elif worst >= 1 - INK_RESIDUAL_MAX:
                status = 'kept'
            else:
                status = 'partly removed'
            box_rows.append({'file': rel, 'phi_category': b.phi_category, 'is_phi': b.is_phi,
                             'residual': worst, 'status': status})
        header = _header_text(pydicom.dcmread(dst, stop_before_pixels=True))
        leaked = sorted({b.phi_category for b in boxes[boxes['is_phi']].itertuples()
                         if any(tok.lower() in header for tok in _header_tokens(b.phi_category, b.phi_value))})
        file_rows.append({'file': rel, 'status': 'checked', 'header_leaks': ', '.join(leaked)})
    boxes_df = pd.DataFrame(box_rows, columns=['file', 'phi_category', 'is_phi', 'residual', 'status'])
    boxes_df['is_phi'] = boxes_df['is_phi'].astype(bool)
    files_df = pd.DataFrame(file_rows, columns=['file', 'status', 'header_leaks'])
    print(f'Checked {len(sops):,} files in {time.perf_counter() - t0:,.1f} s')
    return boxes_df, files_df


if not RUN_ACCURACY:
    print('RUN_ACCURACY is off.')
elif not GT_CSVS:
    print('No phi_overlays.csv found in the DICOM input: accuracy check skipped.')
elif not DICOM_RESULT or not RUN_DIR:
    print('No DICOM run to check.')
else:
    evaluated = evaluate_accuracy(DICOM_RESULT, DICOM_PROFILE, GT_CSVS)
    if evaluated:
        BOXES, FILES = evaluated
        checked = FILES[FILES['status'] == 'checked']
        phi = BOXES[BOXES['is_phi']]
        non_phi = BOXES[~BOXES['is_phi']]
        clean = phi.groupby('file')['status'].apply(lambda s: (s == 'removed').all())
        ACCURACY = {
            'files_with_ground_truth': len(FILES),
            'files_checked': len(checked),
            'files_not_in_output': int((FILES['status'] != 'checked').sum()),
            'phi_boxes': len(phi),
            'phi_boxes_removed_pct': 100 * (phi['status'] == 'removed').mean() if len(phi) else None,
            'phi_boxes_partly_removed_pct': 100 * (phi['status'] == 'partly removed').mean() if len(phi) else None,
            'files_pixel_clean_pct': 100 * clean.mean() if len(clean) else None,
            'non_phi_boxes': len(non_phi),
            'non_phi_kept_pct': 100 * (non_phi['status'] == 'kept').mean() if len(non_phi) else None,
            'files_with_header_leaks': int((checked['header_leaks'].fillna('') != '').sum()) if len(checked) else 0,
        }
        print('=== Accuracy ===')
        display(pd.Series(ACCURACY, name='value').to_frame().style.format('{:,.1f}', na_rep=''))
        if ACCURACY['files_not_in_output']:
            print(f"⚠️  {ACCURACY['files_not_in_output']:,} of {len(FILES):,} ground-truth files are not in the "
                  'output (review queue or error): see the error table in Cell 13.')
            display(FILES[FILES['status'] != 'checked'].head(15))
    if evaluated and len(BOXES):
        by_cat = BOXES.groupby(['is_phi', 'phi_category']).agg(
            boxes=('status', 'size'),
            removed_pct=('status', lambda s: 100 * (s == 'removed').mean()),
            partly_pct=('status', lambda s: 100 * (s == 'partly removed').mean()),
            kept_pct=('status', lambda s: 100 * (s == 'kept').mean()),
            median_residual=('residual', 'median'),
        ).reset_index().sort_values(['is_phi', 'phi_category'], ascending=[False, True])
        print('By category (PHI should be removed; non-PHI should be kept):')
        display(by_cat.style.format({'removed_pct': '{:,.1f}', 'partly_pct': '{:,.1f}', 'kept_pct': '{:,.1f}',
                                     'median_residual': '{:,.3f}'}))
        missed = phi[phi['status'] != 'removed'].sort_values('residual', ascending=False)
        if len(missed):
            print('PHI boxes not fully removed (worst first):')
            display(missed.head(15))
        if ACCURACY['files_with_header_leaks']:
            print('Files whose output header still contains a ground-truth value:')
            display(checked[checked['header_leaks'].fillna('') != ''].head(15))
        bar_chart(list(by_cat['phi_category'] + np.where(by_cat['is_phi'], ' (PHI)', ' (keep)')),
                  list(np.where(by_cat['is_phi'], by_cat['removed_pct'], by_cat['kept_pct'])),
                  'PHI boxes removed / non-PHI boxes kept', '% of boxes', '{:,.0f}%')

if RUN_VERIFY and RUN_DIR:
    from monai_aegis.verify import verify_run
    t0 = time.perf_counter()
    VERIFY = verify_run(RUN_DIR)
    print(f"\n=== PS3.15 verification: {VERIFY['status'].upper()} ({time.perf_counter() - t0:,.1f} s) ===")
    print(VERIFY['totals'])
    if VERIFY['findings']:
        f = pd.DataFrame(VERIFY['findings'])
        display(f.groupby(['check', 'severity']).size().rename('findings').reset_index())
    ACCURACY['verify_status'] = VERIFY['status']
    ACCURACY['verify_failures'] = VERIFY['totals']['failures']
    ACCURACY['verify_warnings'] = VERIFY['totals']['warnings']

---
## ---------- PART B: JPEG / PNG IMAGE ---------------

## Cell 19 — Find, sample and profile image input

In [ ]:
IMAGE_FOLDER    = config['paths']['image_folder']
IMAGE_INPUT_DIR = os.path.join(DRIVE_DEMO_DIR, IMAGE_FOLDER)

img_all = list_images(IMAGE_INPUT_DIR) if os.path.isdir(IMAGE_INPUT_DIR) else []
print('Found {:,} image file(s) in {}'.format(len(img_all), IMAGE_INPUT_DIR))
img_files = pick_sample(img_all, BENCH_MAX_FILES, SAMPLE_SEED)
if len(img_files) < len(img_all):
    print('Sample: {:,} files'.format(len(img_files)))

IMAGE_STAGE   = prepare_inputs(IMAGE_FOLDER, IMAGE_INPUT_DIR, img_files) if img_files else {}
RUN_IMAGE_DIR = os.path.join(RUN_INPUT_ROOT, IMAGE_FOLDER)
IMAGE_PROFILE = profile_images(RUN_IMAGE_DIR, img_files)
show_image_profile(IMAGE_PROFILE)

print()
for f in img_files[:20]:
    print('  {:9.1f} KB  {}'.format((_size(os.path.join(RUN_IMAGE_DIR, f)) or 0) / 1024, f))
if len(img_files) > 20:
    print('  … and {:,} more'.format(len(img_files) - 20))
if not img_files:
    print('No image files found. Add .jpg/.png files to Drive and re-run.')

## Cell 20 — Preview BEFORE image (PHI visible)

In [ ]:
if not img_files:
    raise RuntimeError('No image files found. Run Cell 19 first.')

IMG_SAMPLE      = img_files[0]
img_sample_path = os.path.join(RUN_IMAGE_DIR, IMG_SAMPLE)
print('Using:', IMG_SAMPLE)

orig_img = np.array(Image.open(img_sample_path).convert('RGB'))
plt.figure(figsize=(10, 8))
plt.imshow(orig_img)
plt.title('BEFORE — {} (PHI visible)'.format(IMG_SAMPLE), fontsize=13, color='red')
plt.axis('off'); plt.tight_layout(); plt.show()

## Cell 21 — Run the image benchmark

In [ ]:
print('Running image_runner...\n')
IMAGE_RUNS = []
if RUN_IMAGE and img_files:
    for r in range(1, REPEATS + 1):
        IMAGE_RUNS.append(bench_run(f'image_series_{r}', 'image', image_runner.run_series, RUN_INPUT_ROOT, IMAGE_PROFILE))
IMAGE_RESULT = IMAGE_RUNS[0] if IMAGE_RUNS else None
IMG_RUN_DIR = IMAGE_RESULT['run_dir'] if IMAGE_RESULT else None
print('\n✅  Image runner complete' if IMAGE_RESULT else 'Image benchmark skipped.')

## Cell 22 — Image results and file write measurements

In [ ]:
for res in IMAGE_RUNS:
    show_run(res)
    show_writes(res)

## Cell 23 — Image redaction stats + before / after

In [ ]:
if not IMAGE_RESULT:
    raise RuntimeError('No image run. Run Cell 21 first.')

exts = ('*.jpg', '*.jpeg', '*.png')
output_img_files = sorted(p for e in exts for p in glob.glob(os.path.join(IMG_RUN_DIR, '**', e), recursive=True)) if IMG_RUN_DIR else []
quarantined_img = sorted(p for e in exts for p in glob.glob(os.path.join(IMAGE_RESULT['review_dir'], '**', e), recursive=True))

print('=== Image Pipeline Run Summary ===')
print('  Output files     :', len(output_img_files))
print('  Quarantined files:', len(quarantined_img))
print('  Run folder       :', IMG_RUN_DIR)
if quarantined_img:
    print('\n⚠️  Quarantined (low-confidence / invalid dimensions):')
    for f in quarantined_img[:20]:
        print('    {}'.format(os.path.relpath(f, IMAGE_RESULT['review_dir'])))
    if len(quarantined_img) > 20:
        print('    … and {:,} more'.format(len(quarantined_img) - 20))
else:
    print('\n✅  No files quarantined')

# Output is PNG by default, so match on the name without extension.
stem = os.path.splitext(os.path.basename(IMG_SAMPLE))[0]
SCRUBBED_IMG_PATH = next((f for f in output_img_files if os.path.splitext(os.path.basename(f))[0] == stem),
                         output_img_files[0] if output_img_files else None)

if SCRUBBED_IMG_PATH:
    before_arr = np.array(Image.open(img_sample_path).convert('RGB'))
    after_arr = np.array(Image.open(SCRUBBED_IMG_PATH).convert('RGB'))
    if before_arr.shape == after_arr.shape:
        diff = np.any(before_arr != after_arr, axis=2)
        redacted_px, total_px = int(diff.sum()), diff.size
        redacted_pct = round(100 * redacted_px / total_px, 2)
        print('\n=== Pixel-level Redaction Stats — {} ==='.format(IMG_SAMPLE))
        print('  Image size       : {}×{}'.format(before_arr.shape[1], before_arr.shape[0]))
        print('  Total pixels     : {:,}'.format(total_px))
        print('  Redacted pixels  : {:,}  ({}%)'.format(redacted_px, redacted_pct))
        print('  Preserved pixels : {:,}  ({}%)'.format(total_px - redacted_px, round(100 - redacted_pct, 2)))
        if redacted_px == 0:
            print('\n  ℹ️  No pixels changed — either no PHI detected or file unchanged')
        else:
            print('\n  ✅  PHI regions zeroed, diagnostic content preserved')
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    axes[0].imshow(before_arr)
    axes[0].set_title('BEFORE — PHI visible', fontsize=14, color='red', fontweight='bold')
    axes[0].axis('off')
    axes[1].imshow(after_arr)
    axes[1].set_title('AFTER — PHI redacted by Aegis', fontsize=14, color='green', fontweight='bold')
    axes[1].axis('off')
    plt.suptitle('Aegis De-identification  |  {}'.format(IMG_SAMPLE), fontsize=14, fontweight='bold')
    plt.tight_layout(); plt.show()
else:
    print('\nNo output image found — check runner output above.')

# Ground-truth report for the image run: same flag, same CSV. Images have no DICOM header,
# so there is no aegis_tag_actions.csv.
if IMG_RUN_DIR:
    pixel_csv = os.path.join(IMG_RUN_DIR, 'aegis_pixel_detections.csv')
    if os.path.exists(pixel_csv):
        pix = pd.read_csv(pixel_csv)
        print('aegis_pixel_detections.csv rows:', len(pix))
        if len(pix):
            print('decisions:', pix['decision'].value_counts().to_dict())
        display(pix.head(10))
    else:
        print('  (not found — is reporting.save_ground_truth enabled?)')

---
## ---------- PART C: SUMMARY ---------------

## Cell 24 — All runs side by side

In [ ]:
SUMMARY_COLS = ['kind', 'status', 'input_files', 'items', 'frames', 'input_mb', 'wall_s', 'model_load_s',
                'discovery_s', 'processing_s', 'files_per_s', 'frames_per_s', 's_per_file', 'item_p90_s',
                'ocr_calls', 'cpu_cores_busy', 'peak_rss_mb', 'peak_gpu_mem_mb', 'errors', 'review_files',
                'output_mb', 'file_write_s', 'save_s', 'report_write_s']
RUN_SUMMARY = pd.DataFrame([r['summary'] for r in RESULTS.values()])
RUN_SUMMARY = RUN_SUMMARY.set_index('run') if len(RUN_SUMMARY) else RUN_SUMMARY
if len(RUN_SUMMARY):
    display(RUN_SUMMARY[[c for c in SUMMARY_COLS if c in RUN_SUMMARY]].T)
    bar_chart(list(RUN_SUMMARY.index), list(RUN_SUMMARY['s_per_file'].fillna(0)),
              'Processing time per file', 'seconds per file', '{:,.2f} s')

## Cell 25 — Report-write strategy test (file write measurements, part 3)

Aegis versions with crash-safe reports write each file's report rows as soon as that file finishes. Older versions held every row in memory and wrote once at the end. This test simulates `REPORT_BENCH_FILES` files with the row counts this run actually produced, and times three strategies on the benchmark disk:

| Strategy | Survives a crash | Memory held |
|---|---|---|
| After each file | everything finished so far | one file's rows |
| After each series (`REPORT_SERIES_SIZE` files) | all completed series | one series' rows |
| Once at the end | nothing | every row of the run |

In [ ]:
import tracemalloc


def _row_template(path, fields):
    """A real report row from this run (realistic field widths), or a placeholder."""
    if path and os.path.exists(path):
        df = pd.read_csv(path, dtype=str, nrows=1)
        if len(df):
            return {f: df.iloc[0].get(f, '') for f in fields}
    return {f: f'{f}_value_placeholder' for f in fields}


src = DICOM_RESULT or IMAGE_RESULT
src_dir = src['run_dir'] if src else None
n_in = max(1, src['summary']['input_files']) if src else 1
pix_path = os.path.join(src_dir, 'aegis_pixel_detections.csv') if src_dir else None
tag_path = os.path.join(src_dir, 'aegis_tag_actions.csv') if src_dir else None
pix_per_file = (sum(1 for _ in open(pix_path)) - 1) / n_in if pix_path and os.path.exists(pix_path) else 6
tag_per_file = (sum(1 for _ in open(tag_path)) - 1) / n_in if tag_path and os.path.exists(tag_path) else 12
pix_row = _row_template(pix_path, reporting.PIXEL_FIELDS)
tag_row = _row_template(tag_path, reporting.TAG_FIELDS)
P, T = max(1, round(pix_per_file)), max(0, round(tag_per_file))
print(f'Rows per file from this run: {pix_per_file:,.1f} pixel, {tag_per_file:,.1f} tag (using {P} and {T})')


class _Stream:
    """Same pattern as Aegis's streaming reports: writerows, then flush to the OS."""
    def __init__(self, path, fields):
        self.f = open(path, 'w', newline='', encoding='utf-8')
        self.w = csv.DictWriter(self.f, fieldnames=fields, extrasaction='ignore')
        self.w.writeheader()

    def write(self, rows):
        self.w.writerows(rows)
        self.f.flush()

    def close(self):
        self.f.close()


def report_strategy(files_per_write, n_files, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ps = _Stream(os.path.join(out_dir, 'pixel.csv'), reporting.PIXEL_FIELDS)
    ts = _Stream(os.path.join(out_dir, 'tag.csv'), reporting.TAG_FIELDS)
    pend_p, pend_t, writes = [], [], 0
    t0 = time.perf_counter()
    for i in range(1, n_files + 1):
        pend_p.extend([pix_row] * P)
        pend_t.extend([tag_row] * T)
        if i % files_per_write == 0:
            ps.write(pend_p); ts.write(pend_t); writes += 1
            pend_p, pend_t = [], []
    ps.write(pend_p); ts.write(pend_t); ps.close(); ts.close()
    dt = time.perf_counter() - t0
    size = _size(ps.f.name) + _size(ts.f.name)
    shutil.rmtree(out_dir)
    return dt, writes + 1, size


def held_memory_mb(n_files):
    """Memory to hold n_files' rows as dicts (what writing once at the end keeps)."""
    tracemalloc.start()
    held = [dict(pix_row) for _ in range(n_files * P)] + [dict(tag_row) for _ in range(n_files * T)]
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    del held
    return peak / 2**20


tmp = os.path.join(BENCH_DIR, '_report_test')
strategy_rows = []
for label, k in (('after each file', 1), (f'after each series ({REPORT_SERIES_SIZE} files)', REPORT_SERIES_SIZE),
                 ('once at the end', REPORT_BENCH_FILES)):
    best = min((report_strategy(k, REPORT_BENCH_FILES, tmp) for _ in range(3)), key=lambda r: r[0])
    strategy_rows.append({'strategy': label, 'writes': best[1], 'seconds': best[0],
                          'us_per_file': best[0] / REPORT_BENCH_FILES * 1e6, 'csv_mb': best[2] / 2**20,
                          'rows_held_mb': held_memory_mb(min(k, REPORT_BENCH_FILES))})
REPORT_STRATEGY = pd.DataFrame(strategy_rows)
display(REPORT_STRATEGY.style.format({'writes': '{:,}', 'seconds': '{:,.3f}', 'us_per_file': '{:,.1f}',
                                      'csv_mb': '{:,.1f}', 'rows_held_mb': '{:,.2f}'}))
extra = REPORT_STRATEGY['seconds'].iloc[0] - REPORT_STRATEGY['seconds'].iloc[-1]
per_file_s = (DICOM_RESULT or IMAGE_RESULT)['summary']['s_per_file'] if (DICOM_RESULT or IMAGE_RESULT) else None
print(f'Writing after every file costs {extra:,.2f} s more than once at the end, over {REPORT_BENCH_FILES:,} files.')
if per_file_s:
    print(f'For comparison, processing those files takes about {per_file_s * REPORT_BENCH_FILES / 3600:,.1f} h '
          f'({100 * extra / (per_file_s * REPORT_BENCH_FILES):.4f}% of it).')
bar_chart(list(REPORT_STRATEGY['strategy']), list(REPORT_STRATEGY['seconds']),
          f'Report write time for {REPORT_BENCH_FILES:,} files', 'seconds', '{:,.2f} s')

## Cell 26 — Projection for the full dataset

Scales each run's measured time per file to `TARGET_DATASET_FILES`. Model loading is counted once. Finding files (header reads) scales with the file count. Accurate only if the sample looks like the full dataset: check the profile in Cells 10 and 19.

In [ ]:
def duration(seconds):
    if seconds >= 3600:
        return f'{int(seconds // 3600)} h {int(seconds % 3600 // 60)} min'
    if seconds >= 60:
        return f'{seconds / 60:,.0f} min'
    return f'{seconds:,.0f} s'


proj_rows = []
for name, res in RESULTS.items():
    s = res['summary']
    if not s['input_files'] or not s['processing_s']:
        continue
    per_file = (s['processing_s'] + s['discovery_s']) / s['input_files']
    other = s['wall_s'] - s['processing_s'] - s['discovery_s'] - s['model_load_s']
    est_s = TARGET_DATASET_FILES * (per_file + max(other, 0) / s['input_files']) + s['model_load_s']
    proj_rows.append({
        'run': name,
        'measured_files': s['input_files'],
        'seconds_per_file': per_file,
        'estimated_time': duration(est_s),
        'estimated_hours': est_s / 3600,
        'estimated_output_gb': s['output_mb'] / s['input_files'] * TARGET_DATASET_FILES / 1024,
        'estimated_report_mb': s['report_mb'] / s['input_files'] * TARGET_DATASET_FILES,
        'estimated_review_files': s['review_files'] / s['input_files'] * TARGET_DATASET_FILES,
    })
PROJECTION = pd.DataFrame(proj_rows)
if len(PROJECTION):
    print(f'Projected to {TARGET_DATASET_FILES:,} files:')
    display(PROJECTION.set_index('run').style.format({
        'measured_files': '{:,}', 'seconds_per_file': '{:,.3f}', 'estimated_hours': '{:,.1f}',
        'estimated_output_gb': '{:,.1f}', 'estimated_report_mb': '{:,.1f}', 'estimated_review_files': '{:,.0f}'}))
for label, st in (('DICOM', DICOM_STAGE), ('image', IMAGE_STAGE)):
    if st.get('mode') == 'copy' and st.get('mb_per_s') and st.get('files'):
        mb_per_file = st['mb'] / st['files']
        print(f"Copying {TARGET_DATASET_FILES:,} {label} files from the source at the measured "
              f"{st['mb_per_s']:,.1f} MB/s: about {duration(TARGET_DATASET_FILES * mb_per_file / st['mb_per_s'])}")

## Cell 27 — Save the results

Writes everything to `bench_results/` in the benchmark folder (and to Drive on Colab). The files hold timings, sizes, relative file paths and accuracy categories. They never hold header values or OCR text.

| File | Contents |
|---|---|
| `summary.json` | environment, parameters, warm-up, staging, per-run summaries, accuracy |
| `runs.csv` | one row per run |
| `stages.csv` | time per stage, every run |
| `items.csv` | time per item (file or series), every run |
| `file_writes.csv` | every output file write: bytes, ms |
| `resource_samples.csv` | memory / CPU / GPU over time |
| `disk_baseline.csv`, `report_strategy.csv`, `projection.csv` | Cells 8, 25, 26 |
| `accuracy_boxes.csv`, `accuracy_files.csv` | Cell 18, when ground truth was found |

In [ ]:
RESULTS_DIR = os.path.join(BENCH_DIR, 'bench_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

params = {k: v for k, v in globals().items()
          if k.isupper() and isinstance(v, (int, float, str, bool, type(None)))
          and k in ('BENCH_MAX_FILES', 'SAMPLE_SEED', 'STAGE_TO_LOCAL', 'RUN_DICOM', 'RUN_DICOM_SINGLE',
                    'RUN_IMAGE', 'REPEATS', 'NUM_WORKERS', 'QUIET_LOGS', 'SAMPLE_INTERVAL_S',
                    'INK_RESIDUAL_MAX', 'ACCURACY_FRAMES', 'TARGET_DATASET_FILES', 'AEGIS_BRANCH')}
with open(os.path.join(RESULTS_DIR, 'summary.json'), 'w') as f:
    json.dump({'environment': ENV, 'parameters': params, 'warmup': WARMUP,
               'staging': {'dicom': DICOM_STAGE, 'image': IMAGE_STAGE},
               'runs': [r['summary'] for r in RESULTS.values()],
               'accuracy': ACCURACY}, f, indent=2, default=str)

if RESULTS:
    RUN_SUMMARY.to_csv(os.path.join(RESULTS_DIR, 'runs.csv'))
    pd.concat([stage_table(r).assign(run=n) for n, r in RESULTS.items()]).to_csv(
        os.path.join(RESULTS_DIR, 'stages.csv'), index=False)
    pd.concat([items_table(r).assign(run=n) for n, r in RESULTS.items()]).to_csv(
        os.path.join(RESULTS_DIR, 'items.csv'), index=False)
    pd.concat([write_tables(r)[1].assign(run=n) for n, r in RESULTS.items()]).to_csv(
        os.path.join(RESULTS_DIR, 'file_writes.csv'), index=False)
    pd.concat([r['samples'].assign(run=n) for n, r in RESULTS.items()]).to_csv(
        os.path.join(RESULTS_DIR, 'resource_samples.csv'), index=False)
STORAGE.to_csv(os.path.join(RESULTS_DIR, 'disk_baseline.csv'), index=False)
REPORT_STRATEGY.to_csv(os.path.join(RESULTS_DIR, 'report_strategy.csv'), index=False)
PROJECTION.to_csv(os.path.join(RESULTS_DIR, 'projection.csv'), index=False)
if ACCURACY and 'BOXES' in globals():
    BOXES.to_csv(os.path.join(RESULTS_DIR, 'accuracy_boxes.csv'), index=False)
    FILES.to_csv(os.path.join(RESULTS_DIR, 'accuracy_files.csv'), index=False)

print('Saved to', RESULTS_DIR)
for fn in sorted(os.listdir(RESULTS_DIR)):
    print('  {:<24} {:>10,.1f} KB'.format(fn, _size(os.path.join(RESULTS_DIR, fn)) / 1024))

if SAVE_RESULTS_TO_DRIVE:
    drive_dir = os.path.join(DRIVE_DEMO_DIR, 'benchmarks', BENCH_ID)
    shutil.copytree(RESULTS_DIR, drive_dir, dirs_exist_ok=True)
    print('Copied to Drive:', drive_dir)

## Cell 28 — Free disk space (optional)

The staged input copy and the de-identified outputs stay in the benchmark folder so you can inspect them. Set `DELETE_BENCH_DATA = True` and run this cell to delete them. The `bench_results/` folder is kept.

In [ ]:
DELETE_BENCH_DATA = False

if DELETE_BENCH_DATA:
    for sub in ('input', 'runs'):
        p = os.path.join(BENCH_DIR, sub)
        if os.path.isdir(p):
            shutil.rmtree(p)
            print('Deleted', p)
else:
    print('Kept:', BENCH_DIR, f"({dir_stats(BENCH_DIR)['bytes'] / 2**30:,.2f} GB)")

## Thank You